# Notebook 3 — Wake/Sleep Label Preparation and ECG Feature Engineering

## 1. Purpose and Research Context

This notebook prepares the ECG data for the machine-learning and deep-learning stages of the dissertation.

The original research direction considered ECG-based arrhythmia classification. However, inspection of the Dryad dataset confirmed that explicit arrhythmia or diagnostic disease labels are not available. Therefore, the modelling target has been adapted to a physiological-state classification task that is directly supported by the available data.

The target variable used in this notebook is the recorded `Wake_Sleep` state contained in `Blood_Pressure_Sleep_Info.xlsx`.

The objective is to establish a scientifically justified temporal relationship between the preprocessed ECG windows and the recorded Wake/Sleep observations, assign labels only where the temporal correspondence is sufficiently reliable, and prepare validated datasets for subsequent machine-learning and deep-learning experiments.

The notebook therefore focuses on:

- validating the ECG–Wake/Sleep temporal relationship;
- selecting and documenting an appropriate labelling tolerance;
- assigning Wake/Sleep labels without fabricating diagnostic information;
- analysing the resulting class distribution;
- extracting scientifically motivated ECG features for traditional machine-learning models; and
- preparing standardised ECG waveform windows for deep-learning models.

The resulting datasets will be used in the subsequent modelling notebooks.

In [ ]:
# Cell 2 — Import required libraries

import os
import numpy as np
import pandas as pd

from scipy import signal
from scipy.stats import skew, kurtosis

import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# Cell 3 — Verify required outputs from Notebook 2

required_objects = [
    "ecg_windows",
    "standardised_windows",
    "wake_sleep_df",
    "window_feasibility_df"
]

print("Notebook 2 output availability")
print("=" * 55)

for name in required_objects:
    status = "Available" if name in globals() else "Not available"
    print(f"{name:<25} : {status}")

# Inspect key objects if they are available
print("\nObject details")
print("-" * 55)

if "ecg_windows" in globals():
    print(f"ECG windows: {len(ecg_windows):,}")
    if len(ecg_windows) > 0:
        print(f"ECG window keys: {list(ecg_windows[0].keys())}")

if "standardised_windows" in globals():
    print(f"\nStandardised windows: {len(standardised_windows):,}")
    if len(standardised_windows) > 0:
        print(
            f"Standardised window keys: "
            f"{list(standardised_windows[0].keys())}"
        )

if "wake_sleep_df" in globals():
    print(f"\nWake/Sleep records: {len(wake_sleep_df):,}")
    print(
        f"Wake/Sleep columns: "
        f"{wake_sleep_df.columns.tolist()}"
    )

if "window_feasibility_df" in globals():
    print(
        f"\nWindow feasibility records: "
        f"{len(window_feasibility_df):,}"
    )
    print(
        f"Feasibility columns: "
        f"{window_feasibility_df.columns.tolist()}"
    )

Notebook 2 output availability
ecg_windows               : Not available
standardised_windows      : Not available
wake_sleep_df             : Not available
window_feasibility_df     : Not available

Object details
-------------------------------------------------------


In [ ]:
# Cell 4 — Load dataset and Wake/Sleep information

import zipfile
from google.colab import drive, files

# Mount Google Drive
drive.mount("/content/drive")

# Dataset path established in Notebook 2
zip_path = "/content/drive/MyDrive/QMUL_MSc_Dissertation/Per_Participant_Sensor_Data.zip"

# Verify the dataset archive
if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Dataset ZIP not found at:\n{zip_path}"
    )

if not zipfile.is_zipfile(zip_path):
    raise ValueError("The specified dataset file is not a valid ZIP archive.")

print("Dataset ZIP verified successfully.")

# Upload the Wake/Sleep information file
print("\nPlease upload Blood_Pressure_Sleep_Info.xlsx")
uploaded = files.upload()

excel_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith((".xlsx", ".xls"))
]

if not excel_files:
    raise FileNotFoundError(
        "No Excel file was uploaded. Please upload Blood_Pressure_Sleep_Info.xlsx."
    )

wake_sleep_path = excel_files[0]

# Load Wake/Sleep observations
wake_sleep_df = pd.read_excel(wake_sleep_path)

# Create a combined timestamp
wake_sleep_df["DateTime"] = pd.to_datetime(
    wake_sleep_df["Day_Date"].astype(str)
    + " "
    + wake_sleep_df["Time"].astype(str),
    errors="coerce"
)

print("\nWake/Sleep data loaded successfully.")
print(f"Rows: {len(wake_sleep_df):,}")
print(f"Columns: {wake_sleep_df.columns.tolist()}")
print(f"Valid timestamps: {wake_sleep_df['DateTime'].notna().sum():,}")
print(
    f"Participants: "
    f"{wake_sleep_df['ID'].astype(str).str.zfill(3).nunique()}"
)

Mounted at /content/drive
Dataset ZIP verified successfully.

Please upload Blood_Pressure_Sleep_Info.xlsx


Saving Blood_Pressure_Sleep_Info.xlsx to Blood_Pressure_Sleep_Info.xlsx

Wake/Sleep data loaded successfully.
Rows: 1,623
Columns: ['ID', 'Day_Date', 'Time', 'Systolic', 'Diastolic', 'MAP', 'PP', 'HR', 'Wake_Sleep', 'DateTime']
Valid timestamps: 829
Participants: 30


In [ ]:
# Cell 5 — Diagnose Wake/Sleep timestamp parsing

print("Wake/Sleep timestamp diagnostics")
print("=" * 60)

print("\nOriginal column data types:")
print(wake_sleep_df[["ID", "Day_Date", "Time", "Wake_Sleep"]].dtypes)

print("\nMissing values in source columns:")
print(
    wake_sleep_df[
        ["Day_Date", "Time", "Wake_Sleep"]
    ].isna().sum()
)

print("\nValid and invalid combined timestamps:")
print(
    f"Valid DateTime:   {wake_sleep_df['DateTime'].notna().sum():,}"
)
print(
    f"Invalid DateTime: {wake_sleep_df['DateTime'].isna().sum():,}"
)

print("\nExamples of rows with invalid DateTime:")
display(
    wake_sleep_df.loc[
        wake_sleep_df["DateTime"].isna(),
        ["ID", "Day_Date", "Time", "Wake_Sleep", "DateTime"]
    ].head(10)
)

print("\nExamples of successfully parsed rows:")
display(
    wake_sleep_df.loc[
        wake_sleep_df["DateTime"].notna(),
        ["ID", "Day_Date", "Time", "Wake_Sleep", "DateTime"]
    ].head(5)
)

Wake/Sleep timestamp diagnostics

Original column data types:
ID             int64
Day_Date      object
Time          object
Wake_Sleep     int64
dtype: object

Missing values in source columns:
Day_Date      0
Time          0
Wake_Sleep    0
dtype: int64

Valid and invalid combined timestamps:
Valid DateTime:   829
Invalid DateTime: 794

Examples of rows with invalid DateTime:


,ID,Day_Date,Time,Wake_Sleep,DateTime
48,2,21/06/2022,08:16:00,1,NaT
49,2,21/06/2022,08:30:00,1,NaT
50,2,21/06/2022,08:40:00,1,NaT
51,2,21/06/2022,08:41:00,1,NaT
52,2,21/06/2022,08:46:00,1,NaT
53,2,21/06/2022,09:16:00,1,NaT
54,2,21/06/2022,09:46:00,1,NaT
55,2,21/06/2022,10:16:00,1,NaT
56,2,21/06/2022,10:46:00,1,NaT
57,2,21/06/2022,11:16:00,1,NaT



Examples of successfully parsed rows:


,ID,Day_Date,Time,Wake_Sleep,DateTime
0,1,06/06/2022,11:17:00,1,2022-06-06 11:17:00
1,1,06/06/2022,11:17:00,1,2022-06-06 11:17:00
2,1,06/06/2022,11:19:00,1,2022-06-06 11:19:00
3,1,06/06/2022,11:21:00,1,2022-06-06 11:21:00
4,1,06/06/2022,11:29:00,1,2022-06-06 11:29:00


In [ ]:
# Cell 6 — Correct Wake/Sleep timestamp parsing


wake_sleep_df["DateTime"] = pd.to_datetime(
    wake_sleep_df["Day_Date"].astype(str)
    + " "
    + wake_sleep_df["Time"].astype(str),
    format="%d/%m/%Y %H:%M:%S",
    errors="coerce"
)

print("Corrected Wake/Sleep timestamp parsing")
print("=" * 60)

print(f"Total Wake/Sleep records: {len(wake_sleep_df):,}")
print(f"Valid timestamps:         {wake_sleep_df['DateTime'].notna().sum():,}")
print(f"Invalid timestamps:       {wake_sleep_df['DateTime'].isna().sum():,}")

print("\nTimestamp range:")
print(f"Start: {wake_sleep_df['DateTime'].min()}")
print(f"End:   {wake_sleep_df['DateTime'].max()}")

print("\nWake/Sleep distribution:")
print(
    wake_sleep_df["Wake_Sleep"]
    .value_counts()
    .sort_index()
)

print("\nTimestamp parsing validation:")
if wake_sleep_df["DateTime"].isna().sum() == 0:
    print("All Wake/Sleep timestamps parsed successfully.")
else:
    print("Some timestamps still require investigation.")

Corrected Wake/Sleep timestamp parsing
Total Wake/Sleep records: 1,623
Valid timestamps:         1,623
Invalid timestamps:       0

Timestamp range:
Start: 2022-06-06 11:17:00
End:   2023-09-06 12:21:00

Wake/Sleep distribution:
Wake_Sleep
0     297
1    1326
Name: count, dtype: int64

Timestamp parsing validation:
All Wake/Sleep timestamps parsed successfully.


In [ ]:
# Cell 7 — Validate Wake/Sleep temporal coverage by participant

wake_sleep_df["ParticipantID"] = (
    wake_sleep_df["ID"]
    .astype(str)
    .str.zfill(3)
)

participant_summary = (
    wake_sleep_df
    .groupby("ParticipantID")
    .agg(
        Records=("Wake_Sleep", "size"),
        StartTime=("DateTime", "min"),
        EndTime=("DateTime", "max"),
        WakeRecords=("Wake_Sleep", lambda x: (x == 1).sum()),
        SleepRecords=("Wake_Sleep", lambda x: (x == 0).sum())
    )
    .reset_index()
)

print("Wake/Sleep temporal coverage by participant")
print("=" * 75)

print(f"Participants: {len(participant_summary)}")
print(f"Total records: {len(wake_sleep_df):,}")

print("\nOverall date range:")
print(f"Start: {wake_sleep_df['DateTime'].min()}")
print(f"End:   {wake_sleep_df['DateTime'].max()}")

print("\nParticipant-level summary:")
display(participant_summary)

Wake/Sleep temporal coverage by participant
Participants: 30
Total records: 1,623

Overall date range:
Start: 2022-06-06 11:17:00
End:   2023-09-06 12:21:00

Participant-level summary:


,ParticipantID,Records,StartTime,EndTime,WakeRecords,SleepRecords
0,001,48,2022-06-06 11:17:00,2022-06-07 11:09:00,41,7
1,002,54,2022-06-21 08:16:00,2022-06-22 08:46:00,38,16
2,003,49,2022-08-01 10:11:00,2022-08-02 10:41:00,44,5
3,004,49,2022-08-02 13:57:00,2022-08-03 14:00:00,36,13
4,005,50,2022-08-03 12:21:00,2022-08-04 12:21:00,33,17
5,006,50,2022-08-04 09:17:00,2022-08-05 09:24:00,42,8
6,007,55,2023-03-15 10:40:00,2023-03-16 10:40:00,37,18
7,008,53,2023-03-28 11:40:00,2023-03-29 10:40:00,42,11
8,009,57,2023-04-05 10:43:00,2023-04-06 11:14:00,50,7
9,010,48,2023-04-12 16:50:00,2023-04-13 15:50:00,32,16


In [ ]:
# Cell 8 — Identify ECG recordings in the dataset

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    archive_files = zip_ref.namelist()

# Select ECG CSV recordings only
ecg_csv_files = [
    path for path in archive_files
    if "_ECG.csv" in path
]

# Build recording metadata
recordings = []

for path in ecg_csv_files:
    filename = os.path.basename(path)

    # Extract participant ID from folder structure
    participant_id = path.split("/")[1]

    # Extract recording timestamp from filename
    timestamp_text = filename.replace("_ECG.csv", "")

    recordings.append({
        "ParticipantID": participant_id,
        "ECGFile": filename,
        "Path": path,
        "RecordingStart": pd.to_datetime(
            timestamp_text,
            format="%Y_%m_%d__%H_%M_%S",
            errors="coerce"
        )
    })

recordings_df = pd.DataFrame(recordings)

print("ECG recording inventory")
print("=" * 60)

print(f"ECG CSV recordings: {len(recordings_df)}")
print(
    f"Participants with ECG recordings: "
    f"{recordings_df['ParticipantID'].nunique()}"
)

print(
    f"Invalid recording timestamps: "
    f"{recordings_df['RecordingStart'].isna().sum()}"
)

print("\nRecordings per participant:")
print(
    recordings_df["ParticipantID"]
    .value_counts()
    .sort_index()
)

print("\nFirst five ECG recordings:")
display(
    recordings_df[
        ["ParticipantID", "ECGFile", "RecordingStart"]
    ].head()
)

ECG recording inventory
ECG CSV recordings: 58
Participants with ECG recordings: 30
Invalid recording timestamps: 0

Recordings per participant:
ParticipantID
001    3
002    3
003    4
004    3
005    4
006    3
007    3
008    4
009    2
010    1
011    2
012    1
013    1
014    1
015    1
016    1
017    1
018    1
019    3
020    4
021    1
022    1
023    1
024    1
025    2
026    2
027    1
028    1
029    1
030    1
Name: count, dtype: int64

First five ECG recordings:


,ParticipantID,ECGFile,RecordingStart
0,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 10:28:26
1,001,2022_06_06__15_38_43_ECG.csv,2022-06-06 15:38:43
2,001,2022_06_07__10_42_46_ECG.csv,2022-06-07 10:42:46
3,002,2022_06_21__08_03_26_ECG.csv,2022-06-21 08:03:26
4,002,2022_06_21__08_43_49_ECG.csv,2022-06-21 08:43:49


## 2. Data Loading and Wake/Sleep Timestamp Validation

The Wake/Sleep information was loaded from `Blood_Pressure_Sleep_Info.xlsx`. The dataset contains 1,623 observations from 30 participants, with the recorded physiological state represented by the `Wake_Sleep` variable.

During the initial timestamp conversion, automatic date parsing resulted in only 829 valid timestamps because the source dates use a day/month/year format that can be ambiguous to automatic parsers. Inspection of the original columns confirmed that the source fields themselves contained no missing values.

The timestamps were therefore reconstructed explicitly using the `%d/%m/%Y %H:%M:%S` format. Following this correction, all 1,623 observations were successfully converted into valid timestamps.

This explicit parsing step is retained to ensure that the subsequent temporal matching between ECG windows and Wake/Sleep observations is based on correctly interpreted dates and times.

In [ ]:
# Cell 9 — Confirm final Wake/Sleep dataset integrity

print("Final Wake/Sleep dataset validation")
print("=" * 60)

print(f"Total records:        {len(wake_sleep_df):,}")
print(f"Participants:         {wake_sleep_df['ParticipantID'].nunique()}")
print(f"Valid timestamps:     {wake_sleep_df['DateTime'].notna().sum():,}")
print(f"Missing timestamps:   {wake_sleep_df['DateTime'].isna().sum():,}")
print(f"Wake observations:    {(wake_sleep_df['Wake_Sleep'] == 1).sum():,}")
print(f"Sleep observations:   {(wake_sleep_df['Wake_Sleep'] == 0).sum():,}")

assert wake_sleep_df["DateTime"].notna().all()
assert wake_sleep_df["ParticipantID"].nunique() == 30

print("\nValidation passed successfully.")

Final Wake/Sleep dataset validation
Total records:        1,623
Participants:         30
Valid timestamps:     1,623
Missing timestamps:   0
Wake observations:    1,326
Sleep observations:   297

Validation passed successfully.


## 3. ECG Recording Identification

The ECG recordings are stored within the participant-level dataset archive. Notebook 2 established that the archive contains 58 ECG recordings across 30 participants.

For this notebook, the ECG recordings are identified from the archive without extracting the complete dataset. The recording filename provides the recording start timestamp, while the participant folder provides the corresponding participant identifier.

These recording-level timestamps will subsequently be used to reconstruct the temporal coverage of the ECG data and generate fixed-length ECG windows for matching against the Wake/Sleep observations.

In [ ]:
# Cell 11 — Identify ECG recordings in the dataset archive

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    archive_files = zip_ref.namelist()

# Identify ECG CSV files
ecg_csv_files = [
    path for path in archive_files
    if os.path.basename(path).endswith("_ECG.csv")
]

recordings = []

for path in ecg_csv_files:

    filename = os.path.basename(path)

    # Participant ID from the participant folder
    participant_id = os.path.basename(
        os.path.dirname(path)
    )

    # Recording timestamp from filename
    timestamp_text = filename.replace("_ECG.csv", "")

    recording_start = pd.to_datetime(
        timestamp_text,
        format="%Y_%m_%d__%H_%M_%S",
        errors="coerce"
    )

    recordings.append({
        "ParticipantID": participant_id,
        "ECGFile": filename,
        "Path": path,
        "RecordingStart": recording_start
    })

recordings_df = pd.DataFrame(recordings)

print("ECG recording inventory")
print("=" * 60)

print(f"ECG recordings identified: {len(recordings_df):,}")
print(
    f"Participants represented: "
    f"{recordings_df['ParticipantID'].nunique()}"
)
print(
    f"Invalid recording timestamps: "
    f"{recordings_df['RecordingStart'].isna().sum()}"
)

print("\nRecordings per participant:")
print(
    recordings_df["ParticipantID"]
    .value_counts()
    .sort_index()
)

print("\nFirst five recordings:")
display(
    recordings_df[
        ["ParticipantID", "ECGFile", "RecordingStart"]
    ].head()
)

ECG recording inventory
ECG recordings identified: 58
Participants represented: 30
Invalid recording timestamps: 0

Recordings per participant:
ParticipantID
001_Zephyr    3
002_Zephyr    3
003_Zephyr    4
004_Zephyr    3
005_Zephyr    4
006_Zephyr    3
007_Zephyr    3
008_Zephyr    4
009_Zephyr    2
010_Zephyr    1
011_Zephyr    2
012_Zephyr    1
013_Zephyr    1
014_Zephyr    1
015_Zephyr    1
016_Zephyr    1
017_Zephyr    1
018_Zephyr    1
019_Zephyr    3
020_Zephyr    4
021_Zephyr    1
022_Zephyr    1
023_Zephyr    1
024_Zephyr    1
025_Zephyr    2
026_Zephyr    2
027_Zephyr    1
028_Zephyr    1
029_Zephyr    1
030_Zephyr    1
Name: count, dtype: int64

First five recordings:


,ParticipantID,ECGFile,RecordingStart
0,001_Zephyr,2022_06_06__10_28_26_ECG.csv,2022-06-06 10:28:26
1,001_Zephyr,2022_06_06__15_38_43_ECG.csv,2022-06-06 15:38:43
2,001_Zephyr,2022_06_07__10_42_46_ECG.csv,2022-06-07 10:42:46
3,002_Zephyr,2022_06_21__08_03_26_ECG.csv,2022-06-21 08:03:26
4,002_Zephyr,2022_06_21__08_43_49_ECG.csv,2022-06-21 08:43:49


## 4. ECG–Wake/Sleep Participant Alignment

The ECG recordings and Wake/Sleep observations use slightly different participant identifier formats. The Wake/Sleep dataset identifies participants using numeric identifiers such as `001`, whereas the ECG archive uses identifiers such as `001_Zephyr`.

To enable reliable temporal matching, the ECG participant identifiers are standardised to the numeric participant format used by the Wake/Sleep dataset.

This alignment is performed before any timestamp matching so that ECG windows are compared only with Wake/Sleep observations belonging to the same participant.

In [ ]:
# Cell 13 — Standardise participant identifiers

recordings_df["ParticipantID"] = (
    recordings_df["ParticipantID"]
    .str.extract(r"(\d{3})", expand=False)
)

print("Participant identifier alignment")
print("=" * 60)

print(
    f"Unique ECG participants: "
    f"{recordings_df['ParticipantID'].nunique()}"
)

print(
    f"Unique Wake/Sleep participants: "
    f"{wake_sleep_df['ParticipantID'].nunique()}"
)

# Check that every ECG participant exists in Wake/Sleep data
ecg_participants = set(recordings_df["ParticipantID"])
ws_participants = set(wake_sleep_df["ParticipantID"])

missing_from_wakesleep = sorted(
    ecg_participants - ws_participants
)

print(
    f"ECG participants without Wake/Sleep data: "
    f"{len(missing_from_wakesleep)}"
)

if missing_from_wakesleep:
    print("Participants:", missing_from_wakesleep)
else:
    print("All ECG participants have corresponding Wake/Sleep data.")

print("\nIdentifier alignment validated successfully.")

Participant identifier alignment
Unique ECG participants: 30
Unique Wake/Sleep participants: 30
ECG participants without Wake/Sleep data: 0
All ECG participants have corresponding Wake/Sleep data.

Identifier alignment validated successfully.


## 5. ECG Window Definition

The ECG signals are represented using fixed-length 10-second windows sampled at 250 Hz. Each window therefore contains 2,500 ECG samples.

A 50% overlap is applied between consecutive windows, producing a 5-second step between window starts. This segmentation provides sufficient temporal resolution while retaining enough signal information within each individual window for subsequent classification.

The same segmentation configuration established during ECG preprocessing is retained here to ensure consistency between the preprocessing and modelling stages.

Only complete windows within continuous valid ECG segments are considered. This prevents windows from crossing previously identified signal-quality boundaries.

In [ ]:
# Cell 15 — Define ECG segmentation parameters

sampling_frequency = 250
window_duration_seconds = 10
overlap_fraction = 0.50

window_size = int(
    window_duration_seconds * sampling_frequency
)

step_size = int(
    window_size * (1 - overlap_fraction)
)

print("ECG segmentation parameters")
print("=" * 60)

print(f"Sampling frequency: {sampling_frequency} Hz")
print(f"Window duration:    {window_duration_seconds} seconds")
print(f"Samples per window: {window_size:,}")
print(f"Overlap:            {overlap_fraction * 100:.0f}%")
print(f"Step size:          {step_size:,} samples")

ECG segmentation parameters
Sampling frequency: 250 Hz
Window duration:    10 seconds
Samples per window: 2,500
Overlap:            50%
Step size:          1,250 samples


## 5.1 Reconstructing ECG Windows Across Recordings

The ECG recordings are processed individually to reconstruct the fixed-length windows used for subsequent temporal matching.

For each recording, the ECG waveform is read from the dataset archive and its timestamp column is converted to a datetime representation. The existing ECG quality-control procedure is then applied so that only continuous valid signal regions are used.

Within each valid region, complete 10-second windows are generated using the predefined 50% overlap. Each window retains its participant identifier, recording information, start time, end time, centre time, and ECG waveform.

The centre timestamp of each ECG window is used as the reference point for matching the window to the nearest recorded Wake/Sleep observation.

In [ ]:
# Cell 17 — Restore runtime environment and verify ECG archive

# Core imports
import os
import zipfile
import numpy as np
import pandas as pd

# Google Drive
from google.colab import drive

# Mount Drive if required
drive.mount("/content/drive")

# Dataset location
zip_path = (
    "/content/drive/MyDrive/"
    "QMUL_MSc_Dissertation/"
    "Per_Participant_Sensor_Data.zip"
)

# Verify dataset exists
if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Dataset ZIP not found at:\n{zip_path}"
    )

# Verify ZIP integrity
if not zipfile.is_zipfile(zip_path):
    raise ValueError(
        "The dataset file exists but is not a valid ZIP archive."
    )

print("Runtime environment restored successfully.")
print("=" * 60)
print(f"Dataset found: {os.path.basename(zip_path)}")
print(f"Dataset size: {os.path.getsize(zip_path) / (1024**3):.2f} GB")
print("ZIP archive verified successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Runtime environment restored successfully.
Dataset found: Per_Participant_Sensor_Data.zip
Dataset size: 3.93 GB
ZIP archive verified successfully.


In [ ]:
# Cell 18 — Rebuild ECG recording inventory

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    archive_files = zip_ref.namelist()

# Identify ECG CSV recordings
ecg_csv_files = [
    path
    for path in archive_files
    if os.path.basename(path).endswith("_ECG.csv")
]

recordings = []

for path in ecg_csv_files:

    filename = os.path.basename(path)

    # Extract participant identifier from the folder structure
    participant_id_raw = os.path.basename(
        os.path.dirname(path)
    )

    # Convert identifiers such as 001_Zephyr to 001
    participant_id = (
        pd.Series([participant_id_raw])
        .str.extract(r"(\d{3})", expand=False)
        .iloc[0]
    )

    # Extract recording start time from filename
    timestamp_text = filename.replace("_ECG.csv", "")

    recording_start = pd.to_datetime(
        timestamp_text,
        format="%Y_%m_%d__%H_%M_%S",
        errors="coerce"
    )

    recordings.append({
        "ParticipantID": participant_id,
        "ECGFile": filename,
        "Path": path,
        "RecordingStart": recording_start
    })

recordings_df = pd.DataFrame(recordings)

print("ECG recording inventory rebuilt")
print("=" * 60)

print(f"ECG recordings:              {len(recordings_df):,}")
print(
    f"Participants represented:    "
    f"{recordings_df['ParticipantID'].nunique()}"
)
print(
    f"Invalid recording timestamps: "
    f"{recordings_df['RecordingStart'].isna().sum()}"
)

print("\nRecordings per participant:")
print(
    recordings_df["ParticipantID"]
    .value_counts()
    .sort_index()
)

ECG recording inventory rebuilt
ECG recordings:              58
Participants represented:    30
Invalid recording timestamps: 0

Recordings per participant:
ParticipantID
001    3
002    3
003    4
004    3
005    4
006    3
007    3
008    4
009    2
010    1
011    2
012    1
013    1
014    1
015    1
016    1
017    1
018    1
019    3
020    4
021    1
022    1
023    1
024    1
025    2
026    2
027    1
028    1
029    1
030    1
Name: count, dtype: int64


## 5.2 ECG Window Metadata Generation

Before extracting and storing the ECG waveform values, the temporal metadata for the candidate ECG windows are generated across all available recordings.

Each recording is divided into complete 10-second windows using the predefined 50% overlap. Only windows fully contained within continuous quality-valid ECG segments are considered, and each window is represented by its participant identifier, recording identifier, start time, end time, and centre time.

Keeping the window metadata separate from the waveform values reduces memory usage during the temporal labelling stage. The waveform values will be retrieved only for windows that satisfy the final Wake/Sleep matching criteria.

In [ ]:
# Cell 20 — Optimised ECG window extraction test


import time
import numpy as np
import pandas as pd
import zipfile

# Restore segmentation parameters
sampling_frequency = 250
window_duration_seconds = 10
overlap_fraction = 0.50

window_size = int(
    sampling_frequency * window_duration_seconds
)

step_size = int(
    window_size * (1 - overlap_fraction)
)

# Select the first ECG recording
test_recording = recordings_df.iloc[0]

print("Testing optimised ECG processing")
print("=" * 60)
print(f"Participant: {test_recording['ParticipantID']}")
print(f"File:        {test_recording['ECGFile']}")

start_time = time.time()

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    with zip_ref.open(test_recording["Path"]) as file:

        ecg_data = pd.read_csv(
            file,
            usecols=["Time", "EcgWaveform"]
        )

# Explicit timestamp format used by the ECG files
ecg_data["Time"] = pd.to_datetime(
    ecg_data["Time"],
    format="%d/%m/%Y %H:%M:%S.%f",
    errors="coerce"
)

ecg_data["EcgWaveform"] = pd.to_numeric(
    ecg_data["EcgWaveform"],
    errors="coerce"
)

# Remove invalid samples
ecg_data = ecg_data.dropna(
    subset=["Time", "EcgWaveform"]
).reset_index(drop=True)

# Convert waveform to NumPy
values = ecg_data["EcgWaveform"].to_numpy()

# Identify finite samples
finite_mask = np.isfinite(values)

# Identify consecutive identical-value runs using NumPy
changed = np.empty(len(values), dtype=bool)
changed[0] = True
changed[1:] = values[1:] != values[:-1]

run_starts = np.flatnonzero(changed)
run_ends = np.r_[
    run_starts[1:],
    len(values)
]

run_lengths = run_ends - run_starts

# Mark runs of at least one second as invalid
quality_mask = finite_mask.copy()

for start, end, length in zip(
    run_starts,
    run_ends,
    run_lengths
):
    if length >= sampling_frequency:
        quality_mask[start:end] = False

# Count complete windows within valid data
valid_values = values[quality_mask]

candidate_windows = max(
    0,
    (len(valid_values) - window_size) // step_size + 1
)

elapsed = time.time() - start_time

print("\nTest completed")
print("=" * 60)
print(f"Samples loaded:       {len(values):,}")
print(f"Valid samples:        {quality_mask.sum():,}")
print(f"Candidate windows:    {candidate_windows:,}")
print(f"Processing time:      {elapsed:.2f} seconds")

Testing optimised ECG processing
Participant: 001
File:        2022_06_06__10_28_26_ECG.csv

Test completed
Samples loaded:       4,647,250
Valid samples:        4,637,891
Candidate windows:    3,709
Processing time:      30.92 seconds


## 5.3 Efficient Temporal Window Generation

The temporal labelling stage does not require the complete ECG waveform to be loaded into memory. The ECG recording start and end times are sufficient to construct the candidate 10-second window timeline.

Following the segmentation configuration established in Notebook 2, windows begin every 5 seconds, corresponding to 50% overlap between consecutive 10-second windows. The centre timestamp of each window is then compared with the recorded Wake/Sleep observations for the same participant.

This metadata-first approach substantially reduces computational and memory requirements. The complete ECG waveform will only be retrieved after the final temporal labelling criteria have been established.

In [ ]:
# Cell 22 — Check available recording-level temporal metadata

possible_tables = [
    "ecg_overlap_df",
    "usable_recordings",
    "window_feasibility_df"
]

print("Available temporal metadata")
print("=" * 60)

for name in possible_tables:
    if name in globals():
        obj = globals()[name]

        try:
            print(
                f"{name:<25} : Available "
                f"({len(obj):,} rows)"
            )
        except TypeError:
            print(
                f"{name:<25} : Available"
            )
    else:
        print(
            f"{name:<25} : Not available"
        )

Available temporal metadata
ecg_overlap_df            : Not available
usable_recordings         : Not available
window_feasibility_df     : Not available


In [ ]:
# Cell 23 — Efficient ECG recording time reconstruction

import os
import zipfile
import time
import pandas as pd
import numpy as np

# Dataset and ECG parameters

zip_path = (
    "/content/drive/MyDrive/"
    "QMUL_MSc_Dissertation/"
    "Per_Participant_Sensor_Data.zip"
)

SAMPLING_FREQUENCY = 250

recording_time_results = []

print("Efficient ECG recording time reconstruction")
print("=" * 60)

start_total = time.time()

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    for i, row in recordings_df.iterrows():

        try:

            # Recording start time comes from the filename

            filename = row["ECGFile"]

            timestamp_text = filename.replace(
                "_ECG.csv", ""
            )

            ecg_start = pd.to_datetime(
                timestamp_text,
                format="%Y_%m_%d__%H_%M_%S",
                errors="coerce"
            )

            if pd.isna(ecg_start):
                continue


            # Count CSV data rows without parsing timestamps

            sample_count = 0

            with zip_ref.open(row["Path"]) as file:

                for chunk in iter(
                    lambda: file.read(1024 * 1024),
                    b""
                ):
                    sample_count += chunk.count(b"\n")

            # One newline belongs to the header
            sample_count = max(
                0,
                sample_count - 1
            )

            if sample_count < 2:
                continue


            # Calculate recording duration from sampling rate

            duration_seconds = (
                sample_count - 1
            ) / SAMPLING_FREQUENCY

            ecg_end = (
                ecg_start
                + pd.to_timedelta(
                    duration_seconds,
                    unit="s"
                )
            )

            recording_time_results.append({
                "ParticipantID": row["ParticipantID"],
                "ECGFile": row["ECGFile"],
                "Path": row["Path"],
                "ECGStart": ecg_start,
                "ECGEnd": ecg_end,
                "SampleCount": sample_count,
                "DurationMinutes": (
                    duration_seconds / 60
                )
            })

            print(
                f"Processed {i + 1:02d}/"
                f"{len(recordings_df)} recordings",
                end="\r"
            )

        except Exception as e:

            print(
                f"\nERROR: {row['ECGFile']}"
            )
            print(f"Reason: {e}")

recording_times_df = pd.DataFrame(
    recording_time_results
)

elapsed_total = time.time() - start_total

print("\n\nRecording time reconstruction completed")
print("=" * 60)

print(
    f"Recordings successfully processed: "
    f"{len(recording_times_df):,}"
)

print(
    f"Participants represented: "
    f"{recording_times_df['ParticipantID'].nunique()}"
)

print(
    f"Processing time: "
    f"{elapsed_total:.2f} seconds"
)

print("\nRecording duration summary:")
display(
    recording_times_df[
        [
            "ParticipantID",
            "ECGFile",
            "ECGStart",
            "ECGEnd",
            "SampleCount",
            "DurationMinutes"
        ]
    ].head(10)
)

Efficient ECG recording time reconstruction
Processed 58/58 recordings

Recording time reconstruction completed
Recordings successfully processed: 58
Participants represented: 30
Processing time: 124.23 seconds

Recording duration summary:


,ParticipantID,ECGFile,ECGStart,ECGEnd,SampleCount,DurationMinutes
0,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 10:28:26,2022-06-06 15:38:14.996,4647250,309.816600
1,001,2022_06_06__15_38_43_ECG.csv,2022-06-06 15:38:43,2022-06-07 10:13:04.996,16715500,1114.366600
2,001,2022_06_07__10_42_46_ECG.csv,2022-06-07 10:42:46,2022-06-07 12:19:24.996,1449750,96.649933
3,002,2022_06_21__08_03_26_ECG.csv,2022-06-21 08:03:26,2022-06-21 08:43:16.996,597750,39.849933
4,002,2022_06_21__08_43_49_ECG.csv,2022-06-21 08:43:49,2022-06-21 11:52:34.996,2831500,188.766600
5,002,2022_06_21__11_53_12_ECG.csv,2022-06-21 11:53:12,2022-06-22 10:22:03.996,20233000,1348.866600
6,003,2022_08_01__09_38_43_ECG.csv,2022-08-01 09:38:43,2022-08-01 09:42:59.996,64250,4.283267
7,003,2022_08_01__09_59_53_ECG.csv,2022-08-01 09:59:53,2022-08-01 10:06:49.996,104250,6.949933
8,003,2022_08_01__10_06_54_ECG.csv,2022-08-01 10:06:54,2022-08-01 10:06:59.996,1500,0.099933
9,003,2022_08_01__10_08_04_ECG.csv,2022-08-01 10:08:04,2022-08-02 11:00:43.996,22390000,1492.666600


## 5.4 ECG–Wake/Sleep Temporal Overlap

The reconstructed ECG recording intervals are compared with the available Wake/Sleep observations to determine which ECG recordings have a valid temporal overlap with the target labels.

A recording is considered temporally relevant when its recording interval intersects the period covered by the participant's Wake/Sleep observations.

Recordings without temporal overlap are excluded from subsequent window-level label assignment because assigning a Wake/Sleep state outside the observed temporal range would not be scientifically justified.

This step establishes the subset of ECG recordings that can potentially contribute labelled windows to the final modelling datasets.

In [ ]:
# Cell 25 — Identify ECG recordings with Wake/Sleep temporal overlap

print("ECG–Wake/Sleep temporal overlap analysis")
print("=" * 60)


# Use the recording-level time table created in Cell 23


recording_overlap_df = recording_times_df.copy()

# Ensure participant identifiers have a consistent format
recording_overlap_df["ParticipantID"] = (
    recording_overlap_df["ParticipantID"]
    .astype(str)
    .str.extract(r"(\d{3})", expand=False)
)


# Create participant-level Wake/Sleep observation ranges


ws_ranges = (
    wake_sleep_df
    .dropna(subset=["ParticipantID", "DateTime"])
    .groupby("ParticipantID")["DateTime"]
    .agg(
        WakeSleepStart="min",
        WakeSleepEnd="max"
    )
    .reset_index()
)


# Match ECG recordings to participant Wake/Sleep ranges


recording_overlap_df = recording_overlap_df.merge(
    ws_ranges,
    on="ParticipantID",
    how="left"
)


# Determine whether the time intervals overlap


recording_overlap_df["HasDirectOverlap"] = (
    (recording_overlap_df["ECGStart"]
     <= recording_overlap_df["WakeSleepEnd"])
    &
    (recording_overlap_df["ECGEnd"]
     >= recording_overlap_df["WakeSleepStart"])
)


# Calculate the actual temporal overlap


overlap_start = recording_overlap_df[
    ["ECGStart", "WakeSleepStart"]
].max(axis=1)

overlap_end = recording_overlap_df[
    ["ECGEnd", "WakeSleepEnd"]
].min(axis=1)

recording_overlap_df["OverlapMinutes"] = (
    overlap_end - overlap_start
).dt.total_seconds() / 60

# Non-overlapping recordings have zero usable overlap
recording_overlap_df.loc[
    ~recording_overlap_df["HasDirectOverlap"],
    "OverlapMinutes"
] = 0


# Summary


overlapping_count = int(
    recording_overlap_df["HasDirectOverlap"].sum()
)

non_overlapping_count = (
    len(recording_overlap_df)
    - overlapping_count
)

print(
    f"Total ECG recordings:              "
    f"{len(recording_overlap_df):,}"
)

print(
    f"Recordings with direct overlap:     "
    f"{overlapping_count:,}"
)

print(
    f"Recordings without direct overlap:  "
    f"{non_overlapping_count:,}"
)

print(
    f"Participants represented:           "
    f"{recording_overlap_df['ParticipantID'].nunique()}"
)

print("\nFirst overlapping recordings:")

display(
    recording_overlap_df.loc[
        recording_overlap_df["HasDirectOverlap"],
        [
            "ParticipantID",
            "ECGFile",
            "ECGStart",
            "ECGEnd",
            "WakeSleepStart",
            "WakeSleepEnd",
            "OverlapMinutes"
        ]
    ]
    .sort_values(
        ["ParticipantID", "ECGStart"]
    )
    .head(10)
)

ECG–Wake/Sleep temporal overlap analysis
Total ECG recordings:              58
Recordings with direct overlap:     34
Recordings without direct overlap:  24
Participants represented:           30

First overlapping recordings:


,ParticipantID,ECGFile,ECGStart,ECGEnd,WakeSleepStart,WakeSleepEnd,OverlapMinutes
0,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 10:28:26,2022-06-06 15:38:14.996,2022-06-06 11:17:00,2022-06-07 11:09:00,261.249933
1,001,2022_06_06__15_38_43_ECG.csv,2022-06-06 15:38:43,2022-06-07 10:13:04.996,2022-06-06 11:17:00,2022-06-07 11:09:00,1114.366600
2,001,2022_06_07__10_42_46_ECG.csv,2022-06-07 10:42:46,2022-06-07 12:19:24.996,2022-06-06 11:17:00,2022-06-07 11:09:00,26.233333
3,002,2022_06_21__08_03_26_ECG.csv,2022-06-21 08:03:26,2022-06-21 08:43:16.996,2022-06-21 08:16:00,2022-06-22 08:46:00,27.283267
4,002,2022_06_21__08_43_49_ECG.csv,2022-06-21 08:43:49,2022-06-21 11:52:34.996,2022-06-21 08:16:00,2022-06-22 08:46:00,188.766600
5,002,2022_06_21__11_53_12_ECG.csv,2022-06-21 11:53:12,2022-06-22 10:22:03.996,2022-06-21 08:16:00,2022-06-22 08:46:00,1252.800000
9,003,2022_08_01__10_08_04_ECG.csv,2022-08-01 10:08:04,2022-08-02 11:00:43.996,2022-08-01 10:11:00,2022-08-02 10:41:00,1470.000000
12,004,2022_08_02__13_54_45_ECG.csv,2022-08-02 13:54:45,2022-08-03 14:22:56.996,2022-08-02 13:57:00,2022-08-03 14:00:00,1443.000000
16,005,2022_08_03__12_17_03_ECG.csv,2022-08-03 12:17:03,2022-08-04 13:09:18.996,2022-08-03 12:21:00,2022-08-04 12:21:00,1440.000000
19,006,2022_08_04__09_14_28_ECG.csv,2022-08-04 09:14:28,2022-08-05 09:52:04.996,2022-08-04 09:17:00,2022-08-05 09:24:00,1447.000000


## 5.5 Candidate ECG Window Timeline

The ECG recordings with direct temporal overlap are divided into the predefined 10-second windows using a 5-second step, corresponding to the 50% overlap established during ECG preprocessing.

At this stage, only the temporal metadata of each window is generated. The ECG waveform itself is not loaded. Each candidate window is represented by its participant identifier, recording identifier, start time, end time, and centre time.

The centre timestamp is retained as the primary temporal reference for subsequent matching against the observed Wake/Sleep timestamps.

In [ ]:
# Quality-aware candidate ECG window generation

import numpy as np
import pandas as pd
import zipfile
import time

print("Generating quality-aware ECG window timeline")
print("=" * 65)


# Fixed preprocessing parameters from Notebook 2


SAMPLE_RATE = 250
WINDOW_SECONDS = 10
STEP_SECONDS = 5

SAMPLES_PER_WINDOW = (
    SAMPLE_RATE * WINDOW_SECONDS
)

STEP_SAMPLES = (
    SAMPLE_RATE * STEP_SECONDS
)

MIN_CONSTANT_RUN_SAMPLES = SAMPLE_RATE

print(
    f"Sampling frequency: {SAMPLE_RATE} Hz"
)

print(
    f"Window duration: {WINDOW_SECONDS} seconds"
)

print(
    f"Samples per window: {SAMPLES_PER_WINDOW:,}"
)

print(
    f"Step between windows: {STEP_SECONDS} seconds"
)

print(
    f"Quality threshold: "
    f"{MIN_CONSTANT_RUN_SAMPLES:,} samples "
    f"(1 second)"
)



overlapping_recordings = (
    recording_overlap_df[
        recording_overlap_df["HasDirectOverlap"]
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    f"\nOverlapping recordings: "
    f"{len(overlapping_recordings):,}"
)



quality_window_records = []

window_id = 0

processing_start = time.time()

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    for recording_number, (_, recording) in enumerate(
        overlapping_recordings.iterrows(),
        start=1
    ):

        participant_id = str(
            recording["ParticipantID"]
        )

        ecg_file = recording["ECGFile"]

        source_path = recording["Path"]

        print(
            f"Processing recording "
            f"{recording_number}/"
            f"{len(overlapping_recordings)}: "
            f"{participant_id} | {ecg_file}"
        )


        # Read waveform


        with zip_ref.open(source_path) as file:

            waveform_df = pd.read_csv(
                file,
                usecols=["EcgWaveform"]
            )

        values = pd.to_numeric(
            waveform_df["EcgWaveform"],
            errors="coerce"
        ).to_numpy(
            dtype=np.float32
        )

        total_samples = len(values)


        # Identify finite samples


        finite_mask = np.isfinite(values)


        # Identify consecutive constant-value runs


        if total_samples == 0:
            continue

        changed = np.empty(
            total_samples,
            dtype=bool
        )

        changed[0] = True

        if total_samples > 1:

            changed[1:] = (
                values[1:]
                != values[:-1]
            )

        run_starts = np.flatnonzero(
            changed
        )

        run_ends = np.r_[
            run_starts[1:],
            total_samples
        ]

        run_lengths = (
            run_ends
            - run_starts
        )

        # Create quality mask


        quality_mask = finite_mask.copy()

        for start, end, length in zip(
            run_starts,
            run_ends,
            run_lengths
        ):

            if length >= MIN_CONSTANT_RUN_SAMPLES:

                quality_mask[
                    start:end
                ] = False


        # Identify continuous valid segments


        valid_mask = quality_mask

        validity_change = np.empty(
            total_samples,
            dtype=bool
        )

        validity_change[0] = True

        if total_samples > 1:

            validity_change[1:] = (
                valid_mask[1:]
                != valid_mask[:-1]
            )

        segment_starts = np.flatnonzero(
            validity_change
        )

        segment_ends = np.r_[
            segment_starts[1:],
            total_samples
        ]


        recording_window_count = 0

        for segment_start, segment_end in zip(
            segment_starts,
            segment_ends
        ):

            if not valid_mask[
                segment_start
            ]:
                continue

            segment_length = (
                segment_end
                - segment_start
            )

            if segment_length < SAMPLES_PER_WINDOW:
                continue

            # Number of complete windows
            n_windows = (
                (
                    segment_length
                    - SAMPLES_PER_WINDOW
                )
                // STEP_SAMPLES
            ) + 1

            for window_number in range(
                n_windows
            ):

                start_sample = (
                    segment_start
                    + window_number
                    * STEP_SAMPLES
                )

                end_sample = (
                    start_sample
                    + SAMPLES_PER_WINDOW
                )

                quality_window_records.append({
                    "WindowID": window_id,
                    "ParticipantID": participant_id,
                    "ECGFile": ecg_file,
                    "Path": source_path,
                    "StartSample": start_sample,
                    "EndSample": end_sample,
                    "SegmentStartSample": segment_start,
                    "SegmentEndSample": segment_end
                })

                window_id += 1
                recording_window_count += 1

        print(
            f"  Samples: {total_samples:,}"
        )

        print(
            f"  Valid samples: "
            f"{int(valid_mask.sum()):,}"
        )

        print(
            f"  Quality-valid windows: "
            f"{recording_window_count:,}"
        )


# Create candidate window dataframe


candidate_windows_df = pd.DataFrame(
    quality_window_records
)

processing_elapsed = (
    time.time() - processing_start
)


# Recover actual ECG timestamps


candidate_windows_df = candidate_windows_df.merge(
    recording_overlap_df[
        [
            "ParticipantID",
            "ECGFile",
            "ECGStart"
        ]
    ],
    on=[
        "ParticipantID",
        "ECGFile"
    ],
    how="left"
)

candidate_windows_df["StartTime"] = (
    candidate_windows_df["ECGStart"]
    +
    pd.to_timedelta(
        candidate_windows_df["StartSample"] / SAMPLE_RATE,
        unit="s"
    )
)

candidate_windows_df["EndTime"] = (
    candidate_windows_df["ECGStart"]
    +
    pd.to_timedelta(
        candidate_windows_df["EndSample"] / SAMPLE_RATE,
        unit="s"
    )
)

candidate_windows_df["CenterTime"] = (
    candidate_windows_df["StartTime"]
    +
    pd.Timedelta(
        seconds=WINDOW_SECONDS / 2
    )
)


# Final structure


candidate_windows_df = (
    candidate_windows_df[
        [
            "WindowID",
            "ParticipantID",
            "ECGFile",
            "Path",
            "StartSample",
            "EndSample",
            "StartTime",
            "EndTime",
            "CenterTime",
            "SegmentStartSample",
            "SegmentEndSample"
        ]
    ]
    .sort_values("WindowID")
    .reset_index(drop=True)
)


# Validation


print("\nQuality-aware candidate window generation completed")
print("=" * 65)

print(
    f"Candidate ECG windows: "
    f"{len(candidate_windows_df):,}"
)

print(
    f"Unique ECG recordings: "
    f"{candidate_windows_df['ECGFile'].nunique():,}"
)

print(
    f"Unique participants: "
    f"{candidate_windows_df['ParticipantID'].nunique():,}"
)

print(
    f"Duplicate WindowIDs: "
    f"{candidate_windows_df['WindowID'].duplicated().sum():,}"
)

print(
    f"Missing timestamps: "
    f"{candidate_windows_df['StartTime'].isna().sum():,}"
)

print(
    f"Processing time: "
    f"{processing_elapsed:.2f} seconds"
)

print("\nFirst five quality-valid candidate windows:")

display(
    candidate_windows_df.head()
)

Generating quality-aware ECG window timeline
Sampling frequency: 250 Hz
Window duration: 10 seconds
Samples per window: 2,500
Step between windows: 5 seconds
Quality threshold: 250 samples (1 second)

Overlapping recordings: 34
Processing recording 1/34: 001 | 2022_06_06__10_28_26_ECG.csv
  Samples: 4,647,250
  Valid samples: 4,637,891
  Quality-valid windows: 3,693
Processing recording 2/34: 001 | 2022_06_06__15_38_43_ECG.csv
  Samples: 16,715,500
  Valid samples: 16,715,500
  Quality-valid windows: 13,371
Processing recording 3/34: 001 | 2022_06_07__10_42_46_ECG.csv
  Samples: 1,449,750
  Valid samples: 1,442,342
  Quality-valid windows: 1,142
Processing recording 4/34: 002 | 2022_06_21__08_03_26_ECG.csv
  Samples: 597,750
  Valid samples: 587,063
  Quality-valid windows: 459
Processing recording 5/34: 002 | 2022_06_21__08_43_49_ECG.csv
  Samples: 2,831,500
  Valid samples: 2,827,688
  Quality-valid windows: 2,258
Processing recording 6/34: 002 | 2022_06_21__11_53_12_ECG.csv
  Sample

,WindowID,ParticipantID,ECGFile,Path,StartSample,EndSample,StartTime,EndTime,CenterTime,SegmentStartSample,SegmentEndSample
0,0,001,2022_06_06__10_28_26_ECG.csv,Per_Participant_Sensor_Data/001/001_Zephyr/202...,428,2928,2022-06-06 10:28:27.712,2022-06-06 10:28:37.712,2022-06-06 10:28:32.712,428,33550
1,1,001,2022_06_06__10_28_26_ECG.csv,Per_Participant_Sensor_Data/001/001_Zephyr/202...,1678,4178,2022-06-06 10:28:32.712,2022-06-06 10:28:42.712,2022-06-06 10:28:37.712,428,33550
2,2,001,2022_06_06__10_28_26_ECG.csv,Per_Participant_Sensor_Data/001/001_Zephyr/202...,2928,5428,2022-06-06 10:28:37.712,2022-06-06 10:28:47.712,2022-06-06 10:28:42.712,428,33550
3,3,001,2022_06_06__10_28_26_ECG.csv,Per_Participant_Sensor_Data/001/001_Zephyr/202...,4178,6678,2022-06-06 10:28:42.712,2022-06-06 10:28:52.712,2022-06-06 10:28:47.712,428,33550
4,4,001,2022_06_06__10_28_26_ECG.csv,Per_Participant_Sensor_Data/001/001_Zephyr/202...,5428,7928,2022-06-06 10:28:47.712,2022-06-06 10:28:57.712,2022-06-06 10:28:52.712,428,33550


In [ ]:
# Cell 28 — Match ECG windows to nearest Wake/Sleep observations

print("Matching ECG windows to nearest Wake/Sleep observations")
print("=" * 60)


# Prepare Wake/Sleep observations


ws_matching = wake_sleep_df[
    [
        "ParticipantID",
        "DateTime",
        "Wake_Sleep"
    ]
].copy()

# Ensure consistent participant identifiers
ws_matching["ParticipantID"] = (
    ws_matching["ParticipantID"]
    .astype(str)
    .str.extract(r"(\d{3})", expand=False)
)

# Ensure datetime format
ws_matching["DateTime"] = pd.to_datetime(
    ws_matching["DateTime"],
    errors="coerce"
)

# Remove invalid observations
ws_matching = ws_matching.dropna(
    subset=[
        "ParticipantID",
        "DateTime",
        "Wake_Sleep"
    ]
).copy()


# Prepare ECG window metadata


windows_matching = candidate_windows_df[
    [
        "WindowID",
        "ParticipantID",
        "ECGFile",
        "StartTime",
        "EndTime",
        "CenterTime"
    ]
].copy()

windows_matching["ParticipantID"] = (
    windows_matching["ParticipantID"]
    .astype(str)
)

windows_matching["CenterTime"] = pd.to_datetime(
    windows_matching["CenterTime"],
    errors="coerce"
)

windows_matching = windows_matching.dropna(
    subset=[
        "ParticipantID",
        "CenterTime"
    ]
).copy()



ws_matching = ws_matching.sort_values(
    ["DateTime", "ParticipantID"]
).reset_index(drop=True)

windows_matching = windows_matching.sort_values(
    ["CenterTime", "ParticipantID"]
).reset_index(drop=True)



matched_windows_df = pd.merge_asof(
    windows_matching,
    ws_matching,
    left_on="CenterTime",
    right_on="DateTime",
    by="ParticipantID",
    direction="nearest"
)


# Calculate temporal distance


matched_windows_df["DistanceSeconds"] = (
    matched_windows_df["CenterTime"]
    - matched_windows_df["DateTime"]
).abs().dt.total_seconds()

matched_windows_df["DistanceMinutes"] = (
    matched_windows_df["DistanceSeconds"] / 60
)


# Validation summary


matched_count = int(
    matched_windows_df["DateTime"].notna().sum()
)

unmatched_count = (
    len(matched_windows_df)
    - matched_count
)

print(
    f"Candidate ECG windows:       "
    f"{len(candidate_windows_df):,}"
)

print(
    f"Successfully matched:         "
    f"{matched_count:,}"
)

print(
    f"Without matching observation: "
    f"{unmatched_count:,}"
)

print("\nNearest-observation distance statistics:")

display(
    matched_windows_df[
        "DistanceMinutes"
    ].describe()
)

print("\nFirst five matched windows:")

display(
    matched_windows_df[
        [
            "WindowID",
            "ParticipantID",
            "CenterTime",
            "DateTime",
            "Wake_Sleep",
            "DistanceMinutes"
        ]
    ].head()
)

Matching ECG windows to nearest Wake/Sleep observations
Candidate ECG windows:       530,921
Successfully matched:         530,921
Without matching observation: 0

Nearest-observation distance statistics:


,DistanceMinutes
count,530921.000000
mean,13.331895
std,28.607127
min,0.000267
25%,4.210467
50%,8.970467
75%,13.927867
max,423.436933



First five matched windows:


,WindowID,ParticipantID,CenterTime,DateTime,Wake_Sleep,DistanceMinutes
0,0,001,2022-06-06 10:28:32.712,2022-06-06 11:17:00,1,48.454800
1,1,001,2022-06-06 10:28:37.712,2022-06-06 11:17:00,1,48.371467
2,2,001,2022-06-06 10:28:42.712,2022-06-06 11:17:00,1,48.288133
3,3,001,2022-06-06 10:28:47.712,2022-06-06 11:17:00,1,48.204800
4,4,001,2022-06-06 10:28:52.712,2022-06-06 11:17:00,1,48.121467


## 5.6 Wake/Sleep Label Tolerance Analysis

A nearest Wake/Sleep observation is not automatically treated as a valid label. The temporal distance between each ECG window centre and its nearest observed Wake/Sleep measurement is first evaluated at several candidate tolerances.

The candidate tolerances are 1, 5, 10, 15, and 30 minutes. For each tolerance, the proportion of ECG windows receiving a temporally supported label will be calculated.

This analysis is used to balance data availability against temporal reliability. A larger tolerance increases the number of usable windows but also increases the possibility that the assigned Wake/Sleep state does not accurately represent the physiological state during the ECG window.

The final tolerance will therefore be selected based on the observed temporal distribution, label coverage, class balance, and potential risk of temporal label noise rather than simply choosing the tolerance that produces the largest dataset.

In [ ]:
# Cell 30 — Evaluate candidate Wake/Sleep label tolerances

candidate_tolerances = [1, 5, 10, 15, 30]

tolerance_results = []

total_windows = len(matched_windows_df)

for tolerance in candidate_tolerances:

    usable_mask = (
        matched_windows_df["DistanceMinutes"]
        <= tolerance
    )

    usable_count = int(
        usable_mask.sum()
    )

    coverage_percentage = (
        usable_count / total_windows
    ) * 100

    tolerance_results.append({
        "ToleranceMinutes": tolerance,
        "UsableWindows": usable_count,
        "CoveragePercentage": coverage_percentage
    })

tolerance_analysis_df = pd.DataFrame(
    tolerance_results
)

print("Wake/Sleep label tolerance analysis")
print("=" * 60)

display(
    tolerance_analysis_df
)

Wake/Sleep label tolerance analysis


,ToleranceMinutes,UsableWindows,CoveragePercentage
0,1,38053,7.167356
1,5,154868,29.169688
2,10,293763,55.330831
3,15,421175,79.329128
4,30,518399,97.641457


In [ ]:
# Cell 31 — Evaluate class balance across candidate tolerances

class_balance_results = []

for tolerance in candidate_tolerances:

    # Keep only windows within the current tolerance
    usable = matched_windows_df[
        matched_windows_df["DistanceMinutes"] <= tolerance
    ].copy()

    total = len(usable)

    # Count Wake and Sleep observations
    wake_count = int(
        (usable["Wake_Sleep"] == 1).sum()
    )

    sleep_count = int(
        (usable["Wake_Sleep"] == 0).sum()
    )

    # Calculate percentages
    wake_percentage = (
        wake_count / total * 100
        if total > 0 else 0
    )

    sleep_percentage = (
        sleep_count / total * 100
        if total > 0 else 0
    )

    minority_percentage = min(
        wake_percentage,
        sleep_percentage
    )

    class_balance_results.append({
        "ToleranceMinutes": tolerance,
        "UsableWindows": total,
        "WakeWindows": wake_count,
        "SleepWindows": sleep_count,
        "WakePercentage": wake_percentage,
        "SleepPercentage": sleep_percentage,
        "MinorityClassPercentage": minority_percentage
    })

class_balance_df = pd.DataFrame(
    class_balance_results
)

print("Wake/Sleep class balance across candidate tolerances")
print("=" * 60)

display(
    class_balance_df
)

Wake/Sleep class balance across candidate tolerances


,ToleranceMinutes,UsableWindows,WakeWindows,SleepWindows,WakePercentage,SleepPercentage,MinorityClassPercentage
0,1,38053,30954,7099,81.344441,18.655559,18.655559
1,5,154868,124452,30416,80.360049,19.639951,19.639951
2,10,293763,234646,59117,79.875954,20.124046,20.124046
3,15,421175,333768,87407,79.246869,20.753131,20.753131
4,30,518399,361054,157345,69.647897,30.352103,30.352103


In [ ]:
# Cell 32A — Wake/Sleep temporal transition analysis

print("Wake/Sleep temporal transition analysis")
print("=" * 65)


# Prepare chronologically ordered Wake/Sleep observations


ws_transition_df = (
    wake_sleep_df[
        [
            "ParticipantID",
            "DateTime",
            "Wake_Sleep"
        ]
    ]
    .dropna()
    .sort_values(
        ["ParticipantID", "DateTime"]
    )
    .copy()
)


# Calculate time between consecutive observations


ws_transition_df["PreviousDateTime"] = (
    ws_transition_df
    .groupby("ParticipantID")["DateTime"]
    .shift(1)
)

ws_transition_df["PreviousState"] = (
    ws_transition_df
    .groupby("ParticipantID")["Wake_Sleep"]
    .shift(1)
)

ws_transition_df["ObservationGapMinutes"] = (
    ws_transition_df["DateTime"]
    - ws_transition_df["PreviousDateTime"]
).dt.total_seconds() / 60


# Identify observed Wake/Sleep state changes


ws_transition_df["StateChanged"] = (
    ws_transition_df["Wake_Sleep"]
    != ws_transition_df["PreviousState"]
)

transitions_df = ws_transition_df[
    ws_transition_df["StateChanged"]
    & ws_transition_df["PreviousState"].notna()
].copy()

print("\nObservation timing")
print("-" * 65)

print(
    f"Total observations: "
    f"{len(ws_transition_df):,}"
)

print(
    f"Participants: "
    f"{ws_transition_df['ParticipantID'].nunique():,}"
)

print(
    f"Median observation gap: "
    f"{ws_transition_df['ObservationGapMinutes'].median():.2f} minutes"
)

print(
    f"Mean observation gap: "
    f"{ws_transition_df['ObservationGapMinutes'].mean():.2f} minutes"
)

print(
    f"75th percentile gap: "
    f"{ws_transition_df['ObservationGapMinutes'].quantile(0.75):.2f} minutes"
)

print(
    f"Maximum observation gap: "
    f"{ws_transition_df['ObservationGapMinutes'].max():.2f} minutes"
)

print("\nObserved Wake/Sleep transitions")
print("-" * 65)

print(
    f"Total state transitions: "
    f"{len(transitions_df):,}"
)

print(
    f"Wake → Sleep transitions: "
    f"{
        (
            (transitions_df["PreviousState"] == 1)
            &
            (transitions_df["Wake_Sleep"] == 0)
        ).sum()
    :,}"
)

print(
    f"Sleep → Wake transitions: "
    f"{
        (
            (transitions_df["PreviousState"] == 0)
            &
            (transitions_df["Wake_Sleep"] == 1)
        ).sum()
    :,}"
)

print("\nTransition observation gaps")
print("-" * 65)

if len(transitions_df) > 0:

    display(
        transitions_df[
            "ObservationGapMinutes"
        ].describe()
    )



if len(transitions_df) > 0:

    transitions_df["TransitionReferenceTime"] = (
        transitions_df["PreviousDateTime"]
        + (
            transitions_df["DateTime"]
            - transitions_df["PreviousDateTime"]
        ) / 2
    )


    transition_matching = candidate_windows_df[
        [
            "WindowID",
            "ParticipantID",
            "CenterTime"
        ]
    ].copy()

    transition_reference = transitions_df[
        [
            "ParticipantID",
            "TransitionReferenceTime"
        ]
    ].copy()

    transition_matching = transition_matching.sort_values(
        ["CenterTime", "ParticipantID"]
    )

    transition_reference = transition_reference.sort_values(
        ["TransitionReferenceTime", "ParticipantID"]
    )

    nearest_transition_df = pd.merge_asof(
        transition_matching,
        transition_reference,
        left_on="CenterTime",
        right_on="TransitionReferenceTime",
        by="ParticipantID",
        direction="nearest"
    )

    nearest_transition_df["TransitionDistanceMinutes"] = (
        nearest_transition_df["CenterTime"]
        - nearest_transition_df["TransitionReferenceTime"]
    ).abs().dt.total_seconds() / 60



    transition_sensitivity_results = []

    for tolerance in candidate_tolerances:

        usable = matched_windows_df[
            matched_windows_df["DistanceMinutes"] <= tolerance
        ][
            ["WindowID", "ParticipantID"]
        ].copy()

        usable = usable.merge(
            nearest_transition_df[
                [
                    "WindowID",
                    "TransitionDistanceMinutes"
                ]
            ],
            on="WindowID",
            how="left"
        )

        transition_sensitivity_results.append({
            "ToleranceMinutes": tolerance,
            "UsableWindows": len(usable),
            "Within5MinOfTransition": int(
                (
                    usable["TransitionDistanceMinutes"]
                    <= 5
                ).sum()
            ),
            "Within10MinOfTransition": int(
                (
                    usable["TransitionDistanceMinutes"]
                    <= 10
                ).sum()
            ),
            "TransitionProximity5Pct": (
                (
                    usable["TransitionDistanceMinutes"]
                    <= 5
                ).mean() * 100
            ),
            "TransitionProximity10Pct": (
                (
                    usable["TransitionDistanceMinutes"]
                    <= 10
                ).mean() * 100
            )
        })

    transition_sensitivity_df = pd.DataFrame(
        transition_sensitivity_results
    )

    print("\nTransition proximity among candidate labelled windows")
    print("-" * 65)

    display(
        transition_sensitivity_df
    )

else:

    print(
        "\nNo observed Wake/Sleep transitions were identified."
    )

print("\nAnalysis completed.")

Wake/Sleep temporal transition analysis

Observation timing
-----------------------------------------------------------------
Total observations: 1,623
Participants: 30
Median observation gap: 30.00 minutes
Mean observation gap: 26.70 minutes
75th percentile gap: 30.00 minutes
Maximum observation gap: 63.00 minutes

Observed Wake/Sleep transitions
-----------------------------------------------------------------
Total state transitions: 59
Wake → Sleep transitions: 30
Sleep → Wake transitions: 29

Transition observation gaps
-----------------------------------------------------------------


,ObservationGapMinutes
count,59.000000
mean,48.508475
std,17.266045
min,2.000000
25%,30.000000
50%,60.000000
75%,60.000000
max,60.000000



Transition proximity among candidate labelled windows
-----------------------------------------------------------------


,ToleranceMinutes,UsableWindows,Within5MinOfTransition,Within10MinOfTransition,TransitionProximity5Pct,TransitionProximity10Pct
0,1,38053,150,210,0.394187,0.551862
1,5,154868,377,665,0.243433,0.429398
2,10,293763,497,2777,0.169184,0.945320
3,15,421175,2273,4553,0.539681,1.081023
4,30,518399,7071,14145,1.364007,2.728593



Analysis completed.


### 5.6.1 Temporal Transition Sensitivity Analysis

Before selecting the final Wake/Sleep matching tolerance, the temporal structure of the reference observations was examined. The analysis evaluates the typical spacing between observations and the frequency of observed Wake/Sleep state transitions.

This provides an evidence-based assessment of potential label uncertainty around state transitions and supports comparison of candidate matching tolerances before the final tolerance is selected.

## 5.7 Final Wake/Sleep Matching Tolerance

Based on the temporal-distance and class-balance analyses, a ±15-minute tolerance is selected for the primary Wake/Sleep label assignment.

A ±10-minute tolerance would provide only approximately 55% window coverage, whereas increasing the tolerance to ±15 minutes raises coverage to approximately 79% while maintaining a similar Wake/Sleep class distribution.

Although a ±30-minute tolerance would increase coverage to approximately 98%, it introduces a substantially larger temporal gap between the ECG window and the observed Wake/Sleep state. This increases the potential for temporal label noise, particularly around transitions between wake and sleep.

The ±15-minute tolerance therefore provides a practical balance between temporal reliability, usable sample size, and class representation.

This tolerance is applied only to ECG windows whose nearest observed Wake/Sleep timestamp is within 15 minutes. Windows outside this threshold remain unlabelled and are excluded from the supervised modelling datasets.

## 6. Label Validation

Following the selection of a ±15-minute temporal tolerance, ECG windows are assigned Wake/Sleep labels only when their nearest observed Wake/Sleep measurement falls within this tolerance.

The resulting labelled windows are then validated to confirm that:

- every labelled window has a valid Wake/Sleep state;
- no window exceeds the selected temporal tolerance;
- both Wake and Sleep classes are represented;
- participant identifiers remain valid;
- duplicate window identifiers are absent.

This validation is performed before ECG feature engineering to ensure that downstream modelling is based only on temporally supported and structurally valid labels.

In [ ]:
# Cell 34 — Create and validate final Wake/Sleep labels

FINAL_TOLERANCE_MINUTES = 15



labelled_windows_df = matched_windows_df[
    matched_windows_df["DistanceMinutes"]
    <= FINAL_TOLERANCE_MINUTES
].copy()



labelled_windows_df["WakeSleepLabel"] = (
    labelled_windows_df["Wake_Sleep"]
    .astype(int)
)



valid_labels = labelled_windows_df[
    "WakeSleepLabel"
].isin([0, 1])

valid_distances = (
    labelled_windows_df["DistanceMinutes"]
    <= FINAL_TOLERANCE_MINUTES
)

valid_participants = (
    labelled_windows_df["ParticipantID"]
    .notna()
)

duplicate_windows = (
    labelled_windows_df["WindowID"]
    .duplicated()
    .sum()
)

print("Final Wake/Sleep label validation")
print("=" * 60)

print(
    f"Selected tolerance:              "
    f"±{FINAL_TOLERANCE_MINUTES} minutes"
)

print(
    f"Total labelled windows:           "
    f"{len(labelled_windows_df):,}"
)

print(
    f"Valid Wake/Sleep labels:          "
    f"{valid_labels.sum():,}"
)

print(
    f"Windows within tolerance:         "
    f"{valid_distances.sum():,}"
)

print(
    f"Valid participant identifiers:    "
    f"{valid_participants.sum():,}"
)

print(
    f"Duplicate WindowIDs:              "
    f"{duplicate_windows:,}"
)

print("\nLabel distribution:")

display(
    labelled_windows_df[
        "WakeSleepLabel"
    ].value_counts()
    .sort_index()
    .rename(
        index={
            0: "Sleep",
            1: "Wake"
        }
    )
)

# ------------------------------------------------------------
# Overall validation result
# ------------------------------------------------------------

validation_passed = (
    valid_labels.all()
    and valid_distances.all()
    and valid_participants.all()
    and duplicate_windows == 0
)

print("\nValidation result:")
print(
    "PASSED"
    if validation_passed
    else "REQUIRES INVESTIGATION"
)

Final Wake/Sleep label validation
Selected tolerance:              ±15 minutes
Total labelled windows:           421,175
Valid Wake/Sleep labels:          421,175
Windows within tolerance:         421,175
Valid participant identifiers:    421,175
Duplicate WindowIDs:              0

Label distribution:


,count
WakeSleepLabel,
Sleep,87407
Wake,333768



Validation result:
PASSED


In [ ]:
# Cell 35 — Participant-level label coverage validation

participant_label_summary = (
    labelled_windows_df
    .groupby("ParticipantID")
    .agg(
        TotalWindows=("WindowID", "count"),
        WakeWindows=("WakeSleepLabel", "sum"),
        SleepWindows=(
            "WakeSleepLabel",
            lambda x: (x == 0).sum()
        ),
        MeanDistanceMinutes=(
            "DistanceMinutes",
            "mean"
        )
    )
    .reset_index()
)

participant_label_summary["WakePercentage"] = (
    participant_label_summary["WakeWindows"]
    / participant_label_summary["TotalWindows"]
    * 100
)

participant_label_summary["SleepPercentage"] = (
    participant_label_summary["SleepWindows"]
    / participant_label_summary["TotalWindows"]
    * 100
)

print("Participant-level label coverage validation")
print("=" * 60)

print(
    f"Participants with labelled windows: "
    f"{len(participant_label_summary):,}"
)

print(
    f"Total labelled windows: "
    f"{participant_label_summary['TotalWindows'].sum():,}"
)

print("\nParticipant window-count summary:")

display(
    participant_label_summary[
        [
            "TotalWindows",
            "WakeWindows",
            "SleepWindows",
            "MeanDistanceMinutes"
        ]
    ].describe()
)

print("\nParticipant-level label distribution:")

display(
    participant_label_summary[
        [
            "ParticipantID",
            "TotalWindows",
            "WakeWindows",
            "SleepWindows",
            "WakePercentage",
            "SleepPercentage",
            "MeanDistanceMinutes"
        ]
    ].sort_values("ParticipantID")
)

Participant-level label coverage validation
Participants with labelled windows: 30
Total labelled windows: 421,175

Participant window-count summary:


,TotalWindows,WakeWindows,SleepWindows,MeanDistanceMinutes
count,30.000000,30.000000,30.000000,30.000000
mean,14039.166667,11125.600000,2913.566667,7.073918
std,1154.929348,1028.091791,894.687296,0.210465
min,8487.000000,8127.000000,360.000000,6.540756
25%,13957.500000,10691.500000,2518.500000,6.966670
50%,14258.500000,11269.000000,2880.000000,7.144436
75%,14443.500000,11794.750000,3284.000000,7.231687
max,15058.000000,12808.000000,5097.000000,7.313949



Participant-level label distribution:


,ParticipantID,TotalWindows,WakeWindows,SleepWindows,WakePercentage,SleepPercentage,MeanDistanceMinutes
0,001,13373,11177,2196,83.578853,16.421147,7.240250
1,002,14800,10732,4068,72.513514,27.486486,7.117174
2,003,14601,12808,1793,87.720019,12.279981,7.266043
3,004,14323,10339,3984,72.184598,27.815402,7.313949
4,005,14170,9073,5097,64.029640,35.970360,7.209335
5,006,14324,11450,2874,79.935772,20.064228,7.281209
6,007,14283,11011,3272,77.091647,22.908353,7.184629
7,008,13634,10647,2987,78.091536,21.908464,7.024619
8,009,14416,11898,2518,82.533296,17.466704,7.068655
9,010,13622,9230,4392,67.758038,32.241962,7.239138


In [ ]:
# Cell 36 — Final participant-level label validation

LOW_CLASS_THRESHOLD = 10.0

participant_label_summary["PotentialClassImbalance"] = (
    (participant_label_summary["WakePercentage"] < LOW_CLASS_THRESHOLD)
    |
    (participant_label_summary["SleepPercentage"] < LOW_CLASS_THRESHOLD)
)

flagged_participants = participant_label_summary[
    participant_label_summary["PotentialClassImbalance"]
].copy()

print("Final participant-level label validation")
print("=" * 60)

print(
    f"Total participants: "
    f"{len(participant_label_summary):,}"
)

print(
    f"Participants with both classes represented: "
    f"{(
        (participant_label_summary["WakeWindows"] > 0)
        &
        (participant_label_summary["SleepWindows"] > 0)
    ).sum():,}"
)

print(
    f"Participants flagged for low class representation: "
    f"{len(flagged_participants):,}"
)

print("\nValidation checks:")

print(
    "All participants have labelled windows: "
    f"{(
        participant_label_summary["TotalWindows"] > 0
    ).all()}"
)

print(
    "All participants contain Wake observations: "
    f"{(
        participant_label_summary["WakeWindows"] > 0
    ).all()}"
)

print(
    "All participants contain Sleep observations: "
    f"{(
        participant_label_summary["SleepWindows"] > 0
    ).all()}"
)

print("\nParticipants requiring attention:")

if len(flagged_participants) > 0:
    display(
        flagged_participants[
            [
                "ParticipantID",
                "TotalWindows",
                "WakeWindows",
                "SleepWindows",
                "WakePercentage",
                "SleepPercentage"
            ]
        ]
    )
else:
    print("None")

print("\nSection 6 label validation completed.")

Final participant-level label validation
Total participants: 30
Participants with both classes represented: 30
Participants flagged for low class representation: 1

Validation checks:
All participants have labelled windows: True
All participants contain Wake observations: True
All participants contain Sleep observations: True

Participants requiring attention:


,ParticipantID,TotalWindows,WakeWindows,SleepWindows,WakePercentage,SleepPercentage
13,014,8487,8127,360,95.758218,4.241782



Section 6 label validation completed.


## 7. Class Distribution

The final labelled ECG window dataset is examined to quantify the distribution of Wake and Sleep classes after application of the selected ±15-minute temporal tolerance.

Class proportions are reported at both the overall dataset level and participant level. This provides a clear assessment of class imbalance before feature engineering and model development.

The class distribution is reported descriptively rather than corrected at this stage. Any class-balancing strategy, if required, will be considered during model training so that the original labelled dataset remains available for transparent evaluation.

In [ ]:
# Cell 38 — Overall Wake/Sleep class distribution

class_counts = (
    labelled_windows_df["WakeSleepLabel"]
    .value_counts()
    .sort_index()
)

total_labelled = len(
    labelled_windows_df
)

wake_count = int(
    class_counts.get(1, 0)
)

sleep_count = int(
    class_counts.get(0, 0)
)

wake_percentage = (
    wake_count / total_labelled * 100
)

sleep_percentage = (
    sleep_count / total_labelled * 100
)

print("Final Wake/Sleep class distribution")
print("=" * 60)

print(
    f"Total labelled windows: "
    f"{total_labelled:,}"
)

print(
    f"Wake windows:           "
    f"{wake_count:,} "
    f"({wake_percentage:.2f}%)"
)

print(
    f"Sleep windows:          "
    f"{sleep_count:,} "
    f"({sleep_percentage:.2f}%)"
)

print(
    f"Wake-to-Sleep ratio:    "
    f"{wake_count / sleep_count:.2f}:1"
)

print(
    f"Minority class:         "
    f"{'Sleep' if sleep_count < wake_count else 'Wake'}"
)

Final Wake/Sleep class distribution
Total labelled windows: 421,175
Wake windows:           333,768 (79.25%)
Sleep windows:          87,407 (20.75%)
Wake-to-Sleep ratio:    3.82:1
Minority class:         Sleep


In [ ]:
# Cell 39 — Participant-level Wake/Sleep class distribution

participant_distribution = (
    labelled_windows_df
    .groupby("ParticipantID")["WakeSleepLabel"]
    .agg(
        TotalWindows="count",
        WakeWindows="sum",
        SleepWindows=lambda x: (x == 0).sum()
    )
    .reset_index()
)

participant_distribution["WakePercentage"] = (
    participant_distribution["WakeWindows"]
    / participant_distribution["TotalWindows"]
    * 100
)

participant_distribution["SleepPercentage"] = (
    participant_distribution["SleepWindows"]
    / participant_distribution["TotalWindows"]
    * 100
)

print("Participant-level Wake/Sleep distribution")
print("=" * 60)

print(
    f"Participants: "
    f"{len(participant_distribution):,}"
)

print(
    f"Participants with both classes: "
    f"{(
        (participant_distribution["WakeWindows"] > 0)
        &
        (participant_distribution["SleepWindows"] > 0)
    ).sum():,}"
)

print("\nMost Wake-dominant participants:")

display(
    participant_distribution
    .sort_values("SleepPercentage")
    .head(5)
)

print("\nMost Sleep-represented participants:")

display(
    participant_distribution
    .sort_values("SleepPercentage", ascending=False)
    .head(5)
)

Participant-level Wake/Sleep distribution
Participants: 30
Participants with both classes: 30

Most Wake-dominant participants:


,ParticipantID,TotalWindows,WakeWindows,SleepWindows,WakePercentage,SleepPercentage
13,014,8487,8127,360,95.758218,4.241782
2,003,14601,12808,1793,87.720019,12.279981
24,025,13941,12081,1860,86.658059,13.341941
26,027,14591,12431,2160,85.196354,14.803646
10,011,14960,12620,2340,84.358289,15.641711



Most Sleep-represented participants:


,ParticipantID,TotalWindows,WakeWindows,SleepWindows,WakePercentage,SleepPercentage
4,005,14170,9073,5097,64.029640,35.970360
9,010,13622,9230,4392,67.758038,32.241962
3,004,14323,10339,3984,72.184598,27.815402
1,002,14800,10732,4068,72.513514,27.486486
18,019,14444,10629,3815,73.587649,26.412351


## 8. ECG Feature Engineering

The temporally validated Wake/Sleep labels are now linked to candidate ECG windows. The next stage extracts signal-level information from the ECG recordings for supervised modelling.

Each ECG window contains 10 seconds of signal sampled at 250 Hz, corresponding to 2,500 samples per window. Feature engineering will focus on extracting informative signal characteristics while avoiding unnecessary retention of the full raw dataset in memory.

The initial feature set will include time-domain statistical characteristics of the ECG waveform, followed by signal-quality and morphology-related features where computationally and scientifically appropriate.

Feature extraction will be performed consistently across Wake and Sleep windows so that the resulting dataset can be used for subsequent machine-learning and deep-learning experiments.

> **Preprocessing note:** Notebook 2 established and validated a 0.5–40 Hz, 4th-order Butterworth band-pass filtering procedure (zero-phase, `sosfiltfilt`), applied to each continuous quality-valid ECG segment as a whole before windowing. Notebook 3 reproduces this exact methodology: for each recording, the same quality-mask logic used to generate the quality-aware candidate windows (Section 5.5) is reused to reconstruct the continuous quality-valid segments, the validated band-pass filter is applied to each segment as a whole, and the existing 10-second / 2,500-sample / 50%-overlap windows are then sliced from the filtered segment. The nine time-domain features described below are therefore calculated from windows drawn from the filtered ECG waveform, filtered at the segment level rather than independently per window, consistent with Notebook 2.

In [ ]:
# Cell 41 — Inspect raw ECG CSV structure before feature extraction

import zipfile
import pandas as pd

print("Inspecting raw ECG CSV structure")
print("=" * 60)

# Use the first temporally overlapping ECG recording
example_recording = recording_times_df.iloc[0]

print(
    f"Participant: {example_recording['ParticipantID']}"
)

print(
    f"ECG file: {example_recording['ECGFile']}"
)

print(
    f"ZIP path: {example_recording['Path']}"
)



with zipfile.ZipFile(zip_path, "r") as zip_ref:

    with zip_ref.open(example_recording["Path"]) as file:

        example_ecg_df = pd.read_csv(
            file,
            nrows=5
        )

print("\nRaw ECG columns:")
print(
    list(example_ecg_df.columns)
)

print("\nColumn data types:")
print(
    example_ecg_df.dtypes
)

print("\nFirst five rows:")
display(
    example_ecg_df
)

Inspecting raw ECG CSV structure
Participant: 001
ECG file: 2022_06_06__10_28_26_ECG.csv
ZIP path: Per_Participant_Sensor_Data/001/001_Zephyr/2022_06_06__10_28_26_ECG.csv

Raw ECG columns:
['Time', 'EcgWaveform']

Column data types:
Time           object
EcgWaveform     int64
dtype: object

First five rows:


,Time,EcgWaveform
0,06/06/2022 10:28:26.448,179
1,06/06/2022 10:28:26.452,179
2,06/06/2022 10:28:26.456,179
3,06/06/2022 10:28:26.460,179
4,06/06/2022 10:28:26.464,179


In [ ]:
# Cell 42 — Verify labelled ECG window metadata

print("Labelled ECG window metadata")
print("=" * 60)

print(f"Total labelled windows: {len(labelled_windows_df):,}")

print("\nColumns:")
print(
    labelled_windows_df.columns.tolist()
)

print("\nData types:")
print(
    labelled_windows_df.dtypes
)

print("\nFirst five labelled windows:")

display(
    labelled_windows_df.head(5)
)

Labelled ECG window metadata
Total labelled windows: 421,175

Columns:
['WindowID', 'ParticipantID', 'ECGFile', 'StartTime', 'EndTime', 'CenterTime', 'DateTime', 'Wake_Sleep', 'DistanceSeconds', 'DistanceMinutes', 'WakeSleepLabel']

Data types:
WindowID                    int64
ParticipantID              object
ECGFile                    object
StartTime          datetime64[ns]
EndTime            datetime64[ns]
CenterTime         datetime64[ns]
DateTime           datetime64[ns]
Wake_Sleep                  int64
DistanceSeconds           float64
DistanceMinutes           float64
WakeSleepLabel              int64
dtype: object

First five labelled windows:


,WindowID,ParticipantID,ECGFile,StartTime,EndTime,CenterTime,DateTime,Wake_Sleep,DistanceSeconds,DistanceMinutes,WakeSleepLabel
389,389,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:01:56.744,2022-06-06 11:02:06.744,2022-06-06 11:02:01.744,2022-06-06 11:17:00,1,898.256,14.970933,1
390,390,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:01.744,2022-06-06 11:02:11.744,2022-06-06 11:02:06.744,2022-06-06 11:17:00,1,893.256,14.887600,1
391,391,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:06.744,2022-06-06 11:02:16.744,2022-06-06 11:02:11.744,2022-06-06 11:17:00,1,888.256,14.804267,1
392,392,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:11.744,2022-06-06 11:02:21.744,2022-06-06 11:02:16.744,2022-06-06 11:17:00,1,883.256,14.720933,1
393,393,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:16.744,2022-06-06 11:02:26.744,2022-06-06 11:02:21.744,2022-06-06 11:17:00,1,878.256,14.637600,1


In [ ]:
# Cell 43 — Summarise labelled windows by source ECG recording

recording_summary = (
    labelled_windows_df
    .groupby(["ParticipantID", "ECGFile"], as_index=False)
    .agg(
        LabelledWindows=("WindowID", "count"),
        FirstWindowStart=("StartTime", "min"),
        LastWindowEnd=("EndTime", "max"),
        WakeWindows=("WakeSleepLabel", "sum"),
        SleepWindows=(
            "WakeSleepLabel",
            lambda x: (x == 0).sum()
        )
    )
)

recording_summary["WakePercentage"] = (
    recording_summary["WakeWindows"]
    / recording_summary["LabelledWindows"]
    * 100
)

recording_summary["SleepPercentage"] = (
    recording_summary["SleepWindows"]
    / recording_summary["LabelledWindows"]
    * 100
)

print("Labelled ECG recording summary")
print("=" * 60)

print(
    f"Unique source ECG recordings: "
    f"{len(recording_summary):,}"
)

print(
    f"Total labelled windows: "
    f"{recording_summary['LabelledWindows'].sum():,}"
)

print("\nLabelled windows per recording:")
display(
    recording_summary[
        [
            "ParticipantID",
            "ECGFile",
            "LabelledWindows",
            "WakeWindows",
            "SleepWindows"
        ]
    ]
    .sort_values(["ParticipantID", "ECGFile"])
    .head(20)
)

Labelled ECG recording summary
Unique source ECG recordings: 34
Total labelled windows: 421,175

Labelled windows per recording:


,ParticipantID,ECGFile,LabelledWindows,WakeWindows,SleepWindows
0,001,2022_06_06__10_28_26_ECG.csv,3304,3304,0
1,001,2022_06_06__15_38_43_ECG.csv,9591,7395,2196
2,001,2022_06_07__10_42_46_ECG.csv,478,478,0
3,002,2022_06_21__08_03_26_ECG.csv,459,459,0
4,002,2022_06_21__08_43_49_ECG.csv,2258,2258,0
5,002,2022_06_21__11_53_12_ECG.csv,12083,8015,4068
6,003,2022_08_01__10_08_04_ECG.csv,14601,12808,1793
7,004,2022_08_02__13_54_45_ECG.csv,14323,10339,3984
8,005,2022_08_03__12_17_03_ECG.csv,14170,9073,5097
9,006,2022_08_04__09_14_28_ECG.csv,14324,11450,2874


In [ ]:
# Cell 44 — Validate ECG recording timestamp alignment

print("ECG recording timestamp alignment check")
print("=" * 60)

alignment_results = []

# Check the first 10 reconstructed recordings
alignment_test = recording_times_df.head(10)

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    for _, row in alignment_test.iterrows():

        try:
            with zip_ref.open(row["Path"]) as file:

                first_row = pd.read_csv(
                    file,
                    usecols=["Time"],
                    nrows=1
                )

            actual_first_time = pd.to_datetime(
                first_row["Time"].iloc[0],
                format="%d/%m/%Y %H:%M:%S.%f",
                errors="coerce"
            )

            reconstructed_start = row["ECGStart"]

            offset_seconds = (
                actual_first_time
                - reconstructed_start
            ).total_seconds()

            alignment_results.append({
                "ParticipantID": row["ParticipantID"],
                "ECGFile": row["ECGFile"],
                "ReconstructedStart": reconstructed_start,
                "ActualFirstTimestamp": actual_first_time,
                "OffsetSeconds": offset_seconds
            })

        except Exception as e:

            print(
                f"Could not inspect {row['ECGFile']}: {e}"
            )

alignment_df = pd.DataFrame(
    alignment_results
)

print(
    f"Recordings checked: "
    f"{len(alignment_df):,}"
)

print("\nTimestamp alignment results:")

display(
    alignment_df
)

print("\nOffset summary:")

display(
    alignment_df["OffsetSeconds"].describe()
)

ECG recording timestamp alignment check
Recordings checked: 10

Timestamp alignment results:


,ParticipantID,ECGFile,ReconstructedStart,ActualFirstTimestamp,OffsetSeconds
0,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 10:28:26,2022-06-06 10:28:26.448,0.448
1,001,2022_06_06__15_38_43_ECG.csv,2022-06-06 15:38:43,2022-06-06 15:38:43.457,0.457
2,001,2022_06_07__10_42_46_ECG.csv,2022-06-07 10:42:46,2022-06-07 10:42:46.454,0.454
3,002,2022_06_21__08_03_26_ECG.csv,2022-06-21 08:03:26,2022-06-21 08:03:26.421,0.421
4,002,2022_06_21__08_43_49_ECG.csv,2022-06-21 08:43:49,2022-06-21 08:43:49.425,0.425
5,002,2022_06_21__11_53_12_ECG.csv,2022-06-21 11:53:12,2022-06-21 11:53:12.425,0.425
6,003,2022_08_01__09_38_43_ECG.csv,2022-08-01 09:38:43,2022-08-01 09:38:43.430,0.430
7,003,2022_08_01__09_59_53_ECG.csv,2022-08-01 09:59:53,2022-08-01 09:59:53.426,0.426
8,003,2022_08_01__10_06_54_ECG.csv,2022-08-01 10:06:54,2022-08-01 10:06:54.425,0.425
9,003,2022_08_01__10_08_04_ECG.csv,2022-08-01 10:08:04,2022-08-01 10:08:04.428,0.428



Offset summary:


,OffsetSeconds
count,10.000000
mean,0.433900
std,0.013552
min,0.421000
25%,0.425000
50%,0.427000
75%,0.443500
max,0.457000


In [ ]:
# Cell 45 — Calculate ECG timestamp offsets for all recordings

print("Calculating ECG timestamp offsets")
print("=" * 60)

all_alignment_results = []

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    for _, row in recording_times_df.iterrows():

        try:
            with zip_ref.open(row["Path"]) as file:

                first_row = pd.read_csv(
                    file,
                    usecols=["Time"],
                    nrows=1
                )

            actual_first_time = pd.to_datetime(
                first_row["Time"].iloc[0],
                format="%d/%m/%Y %H:%M:%S.%f",
                errors="coerce"
            )

            reconstructed_start = row["ECGStart"]

            offset_seconds = (
                actual_first_time
                - reconstructed_start
            ).total_seconds()

            all_alignment_results.append({
                "ParticipantID": row["ParticipantID"],
                "ECGFile": row["ECGFile"],
                "Path": row["Path"],
                "ReconstructedStart": reconstructed_start,
                "ActualFirstTimestamp": actual_first_time,
                "OffsetSeconds": offset_seconds
            })

        except Exception as e:

            print(
                f"Error reading {row['ECGFile']}: {e}"
            )

ecg_alignment_df = pd.DataFrame(
    all_alignment_results
)

print(
    f"Recordings checked: "
    f"{len(ecg_alignment_df):,}"
)

print("\nOffset statistics:")

display(
    ecg_alignment_df[
        "OffsetSeconds"
    ].describe()
)

print("\nOffset range:")

print(
    f"Minimum: "
    f"{ecg_alignment_df['OffsetSeconds'].min():.4f} seconds"
)

print(
    f"Maximum: "
    f"{ecg_alignment_df['OffsetSeconds'].max():.4f} seconds"
)

print(
    f"Mean: "
    f"{ecg_alignment_df['OffsetSeconds'].mean():.4f} seconds"
)

print(
    f"Standard deviation: "
    f"{ecg_alignment_df['OffsetSeconds'].std():.4f} seconds"
)

Calculating ECG timestamp offsets
Recordings checked: 58

Offset statistics:


,OffsetSeconds
count,58.000000
mean,0.438483
std,0.011117
min,0.421000
25%,0.427250
50%,0.440000
75%,0.447000
max,0.460000



Offset range:
Minimum: 0.4210 seconds
Maximum: 0.4600 seconds
Mean: 0.4385 seconds
Standard deviation: 0.0111 seconds


## 8.1 ECG Timestamp Alignment

The raw ECG timestamps were compared with the reconstructed recording start times for all 58 ECG recordings.

A consistent sub-second offset was observed across the recordings. The mean offset was approximately 0.44 seconds, with values ranging from 0.421 to 0.460 seconds and a standard deviation of approximately 0.011 seconds.

The small and consistent offset indicates stable timestamp alignment across the recordings rather than recording-specific timing errors. The measured recording-specific offsets are therefore retained as metadata and accounted for during subsequent ECG waveform extraction.

In [ ]:
# Cell 47 — Test extraction of one labelled ECG window

import numpy as np

print("Testing single ECG window extraction")
print("=" * 60)



test_window = labelled_windows_df.iloc[0]

participant_id = test_window["ParticipantID"]
ecg_file = test_window["ECGFile"]
window_start = test_window["StartTime"]
window_end = test_window["EndTime"]
window_label = test_window["WakeSleepLabel"]

print(
    f"Participant: {participant_id}"
)

print(
    f"ECG file: {ecg_file}"
)

print(
    f"Window start: {window_start}"
)

print(
    f"Window end: {window_end}"
)

print(
    f"Label: "
    f"{'Wake' if window_label == 1 else 'Sleep'}"
)


matching_recording = recording_times_df[
    (recording_times_df["ParticipantID"] == participant_id)
    &
    (recording_times_df["ECGFile"] == ecg_file)
]

if len(matching_recording) != 1:
    raise ValueError(
        "Could not uniquely identify the source ECG recording."
    )

source_path = matching_recording.iloc[0]["Path"]


# Read only the required portion of the ECG CSV


with zipfile.ZipFile(zip_path, "r") as zip_ref:

    with zip_ref.open(source_path) as file:

        ecg_sample = pd.read_csv(
            file,
            usecols=["Time", "EcgWaveform"]
        )


# Parse timestamps and waveform


ecg_sample["Time"] = pd.to_datetime(
    ecg_sample["Time"],
    format="%d/%m/%Y %H:%M:%S.%f",
    errors="coerce"
)

ecg_sample["EcgWaveform"] = pd.to_numeric(
    ecg_sample["EcgWaveform"],
    errors="coerce"
)

ecg_sample = ecg_sample.dropna(
    subset=["Time", "EcgWaveform"]
)



window_data = ecg_sample[
    (ecg_sample["Time"] >= window_start)
    &
    (ecg_sample["Time"] < window_end)
].copy()

signal = (
    window_data["EcgWaveform"]
    .to_numpy(dtype=np.float32)
)


print("\nExtraction result")
print("-" * 60)

print(
    f"Samples extracted: "
    f"{len(signal):,}"
)

print(
    f"Expected samples: "
    f"2,500"
)

print(
    f"Missing/invalid samples: "
    f"{np.isnan(signal).sum():,}"
)

if len(signal) > 0:

    print(
        f"Signal minimum: "
        f"{signal.min():.2f}"
    )

    print(
        f"Signal maximum: "
        f"{signal.max():.2f}"
    )

    print(
        f"Signal mean: "
        f"{signal.mean():.2f}"
    )

    print(
        f"Signal standard deviation: "
        f"{signal.std():.2f}"
    )

print("\nSingle-window extraction test completed.")

Testing single ECG window extraction
Participant: 001
ECG file: 2022_06_06__10_28_26_ECG.csv
Window start: 2022-06-06 11:01:56.744000
Window end: 2022-06-06 11:02:06.744000
Label: Wake

Extraction result
------------------------------------------------------------
Samples extracted: 2,500
Expected samples: 2,500
Missing/invalid samples: 0
Signal minimum: 1968.00
Signal maximum: 2117.00
Signal mean: 2040.85
Signal standard deviation: 15.54

Single-window extraction test completed.


In [ ]:
# Cell 48 — Validate extracted ECG window timing

print("Extracted ECG window timing validation")
print("=" * 60)

if len(window_data) > 0:

    actual_start = window_data["Time"].iloc[0]
    actual_end = window_data["Time"].iloc[-1]

    actual_duration = (
        actual_end - actual_start
    ).total_seconds()

    sample_intervals = (
        window_data["Time"]
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    print(f"Requested window start: {window_start}")
    print(f"Requested window end:   {window_end}")

    print("\nActual ECG samples:")
    print(f"First sample:            {actual_start}")
    print(f"Last sample:             {actual_end}")

    print("\nTiming:")
    print(
        f"Observed signal duration: "
        f"{actual_duration:.6f} seconds"
    )

    print(
        f"Number of samples:       "
        f"{len(window_data):,}"
    )

    print(
        f"Mean sampling interval:  "
        f"{sample_intervals.mean():.6f} seconds"
    )

    print(
        f"Estimated sampling rate: "
        f"{1 / sample_intervals.mean():.2f} Hz"
    )

    print(
        f"Expected sampling rate:  "
        f"250 Hz"
    )

else:
    print("No ECG samples were extracted.")

print("\nTiming validation completed.")

Extracted ECG window timing validation
Requested window start: 2022-06-06 11:01:56.744000
Requested window end:   2022-06-06 11:02:06.744000

Actual ECG samples:
First sample:            2022-06-06 11:01:56.744000
Last sample:             2022-06-06 11:02:06.740000

Timing:
Observed signal duration: 9.996000 seconds
Number of samples:       2,500
Mean sampling interval:  0.004000 seconds
Estimated sampling rate: 250.00 Hz
Expected sampling rate:  250 Hz

Timing validation completed.


## 8.2 Time-Domain ECG Feature Set

For each validated 10-second ECG window, the continuous quality-valid segment containing the window is first band-pass filtered as a whole using the validated Notebook 2 parameters (0.5–40 Hz, 4th-order Butterworth, zero-phase), and a compact set of time-domain statistical features is then extracted from the corresponding filtered window.

The initial feature set includes the mean, standard deviation, minimum, maximum, range, median, interquartile range, root mean square (RMS), and signal energy.

These features provide information about the central tendency, variability, amplitude range, and overall signal magnitude of each ECG window while keeping the feature representation computationally efficient.

The extracted features will be validated for missing values, infinite values, and numerical stability before being used for machine-learning and deep-learning dataset preparation.

In [ ]:
# Cell 50 — Test ECG feature extraction on one validated window

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt



FILTER_SAMPLE_RATE_HZ = 250
FILTER_LOW_CUTOFF_HZ = 0.5
FILTER_HIGH_CUTOFF_HZ = 40
FILTER_ORDER = 4

# Matches the quality-mask threshold used in Section 5.5 (Cell 27)
MIN_CONSTANT_RUN_SAMPLES = FILTER_SAMPLE_RATE_HZ

# Matches Notebook 2's minimum segment length for sosfiltfilt
MINIMUM_FILTER_LENGTH = 28

_bandpass_sos = butter(
    FILTER_ORDER,
    [FILTER_LOW_CUTOFF_HZ, FILTER_HIGH_CUTOFF_HZ],
    btype="band",
    fs=FILTER_SAMPLE_RATE_HZ,
    output="sos"
)


def apply_segment_level_bandpass_filter(signal_array):
    """
    Reconstruct continuous quality-valid ECG segments within a
    recording, reusing the exact quality-mask logic already
    established in Section 5.5 (Cell 27), and apply the validated
    Notebook 2 band-pass filter to each continuous valid segment
    as a whole.

    This reproduces Notebook 2's validated methodology exactly:

        continuous quality-valid ECG segment
        -> 0.5-40 Hz Butterworth filter (sosfiltfilt)
        -> window extraction

    rather than filtering each 10-second window independently.
    Filtering the full continuous segment before slicing avoids
    the boundary/edge effects that independent window-level
    filtering would introduce every 5 seconds (at each window
    edge), which is especially relevant for a 0.5 Hz low cutoff.

    Returns a filtered array the same length as the input. Samples
    outside continuous valid segments, or inside segments too
    short to filter, are left as NaN. These positions are never
    used because ECG windows are only ever generated within
    continuous quality-valid segments of sufficient length
    (Section 5.5), so every extracted window is guaranteed to be
    fully covered by the filtered output.
    """

    signal_array = np.asarray(
        signal_array,
        dtype=np.float64
    )

    total_samples = len(signal_array)

    filtered_array = np.full(
        total_samples,
        np.nan,
        dtype=np.float64
    )

    if total_samples == 0:
        return filtered_array


    finite_mask = np.isfinite(signal_array)

    changed = np.empty(total_samples, dtype=bool)
    changed[0] = True

    if total_samples > 1:
        changed[1:] = (
            signal_array[1:] != signal_array[:-1]
        )

    run_starts = np.flatnonzero(changed)
    run_ends = np.r_[run_starts[1:], total_samples]
    run_lengths = run_ends - run_starts

    quality_mask = finite_mask.copy()

    for start, end, length in zip(
        run_starts, run_ends, run_lengths
    ):
        if length >= MIN_CONSTANT_RUN_SAMPLES:
            quality_mask[start:end] = False

    valid_mask = quality_mask

    validity_change = np.empty(total_samples, dtype=bool)
    validity_change[0] = True

    if total_samples > 1:
        validity_change[1:] = (
            valid_mask[1:] != valid_mask[:-1]
        )

    segment_starts = np.flatnonzero(validity_change)
    segment_ends = np.r_[segment_starts[1:], total_samples]



    for segment_start, segment_end in zip(
        segment_starts, segment_ends
    ):

        if not valid_mask[segment_start]:
            continue

        segment_values = signal_array[
            segment_start:segment_end
        ]

        if len(segment_values) <= MINIMUM_FILTER_LENGTH:
            continue

        filtered_array[segment_start:segment_end] = sosfiltfilt(
            _bandpass_sos,
            segment_values
        )

    return filtered_array


def extract_time_domain_features(signal):
    """
    Extract compact time-domain features from one ECG window.

    The signal passed in is expected to already be band-pass
    filtered (via apply_segment_level_bandpass_filter, applied to
    the full continuous valid segment before windowing). This
    function performs feature calculation only.
    """

    signal = np.asarray(signal, dtype=np.float32)

    # Remove invalid values
    signal = signal[np.isfinite(signal)]

    if len(signal) == 0:
        return None

    q25, q75 = np.percentile(signal, [25, 75])

    features = {
        "Mean": np.mean(signal),
        "Std": np.std(signal),
        "Min": np.min(signal),
        "Max": np.max(signal),
        "Range": np.max(signal) - np.min(signal),
        "Median": np.median(signal),
        "IQR": q75 - q25,
        "RMS": np.sqrt(np.mean(signal ** 2)),
        "Energy": np.sum(signal ** 2)
    }

    return features




ecg_sample["EcgWaveform_Filtered"] = apply_segment_level_bandpass_filter(
    ecg_sample["EcgWaveform"].to_numpy()
)

filtered_window_signal = (
    ecg_sample.loc[
        window_data.index,
        "EcgWaveform_Filtered"
    ]
    .to_numpy(dtype=np.float32)
)

test_features = extract_time_domain_features(
    filtered_window_signal
)

test_features_df = pd.DataFrame(
    [test_features]
)

print("Single-window ECG feature extraction test")
print("=" * 60)

print(
    f"Samples used: {len(filtered_window_signal):,}"
)

print(
    f"Features extracted: {len(test_features):,}"
)

print("\nExtracted features (segment-filtered ECG waveform):")

display(
    test_features_df
)

# ------------------------------------------------------------
# Feature validation
# ------------------------------------------------------------

numeric_values = test_features_df.to_numpy(
    dtype=np.float64
)

print("\nFeature validation:")

print(
    f"Missing values: "
    f"{np.isnan(numeric_values).sum():,}"
)

print(
    f"Infinite values: "
    f"{np.isinf(numeric_values).sum():,}"
)

print(
    f"All features finite: "
    f"{np.isfinite(numeric_values).all()}"
)


Single-window ECG feature extraction test
Samples used: 2,500
Features extracted: 9

Extracted features (segment-filtered ECG waveform):


,Mean,Std,Min,Max,Range,Median,IQR,RMS,Energy
0,-0.406652,12.306687,-49.314396,73.84021,123.154602,-1.36258,10.110797,12.313403,379049.75



Feature validation:
Missing values: 0
Infinite values: 0
All features finite: True


## 8.3 Efficient ECG Feature Extraction

The validated feature extraction function is now applied to the complete set of labelled ECG windows.

To avoid unnecessary memory usage and repeated access to the large ECG archive, extraction is performed recording-by-recording. For each source ECG recording, only the required waveform data are loaded, the corresponding 10-second windows are identified, and the nine time-domain features are calculated.

The resulting feature records retain the original window identifier, participant identifier, ECG recording information, temporal information, and Wake/Sleep label so that the extracted features remain traceable to their source observations.

Intermediate results are accumulated incrementally rather than storing the complete raw ECG waveform dataset in memory.

In [ ]:
# Cell 52 — Test efficient ECG feature extraction on 3 recordings

import time
import numpy as np
import pandas as pd

BATCH_SIZE = 3

test_recordings = recording_summary.head(BATCH_SIZE)

batch_feature_results = []

batch_start_time = time.time()

print("Testing efficient ECG feature extraction")
print("=" * 60)
print(f"Test recordings: {BATCH_SIZE}")

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    for recording_number, (_, recording) in enumerate(
        test_recordings.iterrows(),
        start=1
    ):

        participant_id = recording["ParticipantID"]
        ecg_file = recording["ECGFile"]

        print(
            f"\nProcessing recording "
            f"{recording_number}/{BATCH_SIZE}: "
            f"Participant {participant_id} | {ecg_file}"
        )

        # Source path
        source_path = recording_times_df.loc[
            (
                (recording_times_df["ParticipantID"] == participant_id)
                &
                (recording_times_df["ECGFile"] == ecg_file)
            ),
            "Path"
        ].iloc[0]

        # Windows belonging to this recording
        recording_windows = labelled_windows_df[
            (labelled_windows_df["ParticipantID"] == participant_id)
            &
            (labelled_windows_df["ECGFile"] == ecg_file)
        ].copy()

        with zip_ref.open(source_path) as file:

            ecg_data = pd.read_csv(
                file,
                usecols=["Time", "EcgWaveform"]
            )

        ecg_data["Time"] = pd.to_datetime(
            ecg_data["Time"],
            format="%d/%m/%Y %H:%M:%S.%f",
            errors="coerce"
        )

        ecg_data["EcgWaveform"] = pd.to_numeric(
            ecg_data["EcgWaveform"],
            errors="coerce"
        )

        ecg_data = ecg_data.dropna(
            subset=["Time", "EcgWaveform"]
        ).reset_index(drop=True)



        ecg_data["EcgWaveform_Filtered"] = apply_segment_level_bandpass_filter(
            ecg_data["EcgWaveform"].to_numpy()
        )

        # Extract features for each labelled window
        for _, window in recording_windows.iterrows():

            window_data = ecg_data[
                (ecg_data["Time"] >= window["StartTime"])
                &
                (ecg_data["Time"] < window["EndTime"])
            ]

            signal_window = (
                window_data["EcgWaveform_Filtered"]
                .to_numpy(dtype=np.float32)
            )

            if len(signal_window) != 2500:
                continue

            features = extract_time_domain_features(
                signal_window
            )

            if features is None:
                continue

            feature_record = {
                "WindowID": window["WindowID"],
                "ParticipantID": participant_id,
                "ECGFile": ecg_file,
                "StartTime": window["StartTime"],
                "EndTime": window["EndTime"],
                "WakeSleepLabel": window["WakeSleepLabel"],
                **features
            }

            batch_feature_results.append(
                feature_record
            )

batch_elapsed = time.time() - batch_start_time

batch_features_df = pd.DataFrame(
    batch_feature_results
)

print("\nBatch test completed")
print("=" * 60)

print(
    f"Feature rows created: "
    f"{len(batch_features_df):,}"
)

print(
    f"Processing time: "
    f"{batch_elapsed:.2f} seconds"
)

if batch_elapsed > 0:
    print(
        f"Average time per recording: "
        f"{batch_elapsed / BATCH_SIZE:.2f} seconds"
    )

print("\nFeature columns:")

print(
    batch_features_df.columns.tolist()
)

print("\nMissing values:")

print(
    batch_features_df.isna().sum().sum()
)

print("\nFirst five feature records:")

display(
    batch_features_df.head()
)

Testing efficient ECG feature extraction
Test recordings: 3

Processing recording 1/3: Participant 001 | 2022_06_06__10_28_26_ECG.csv

Processing recording 2/3: Participant 001 | 2022_06_06__15_38_43_ECG.csv

Processing recording 3/3: Participant 001 | 2022_06_07__10_42_46_ECG.csv

Batch test completed
Feature rows created: 13,372
Processing time: 1598.52 seconds
Average time per recording: 532.84 seconds

Feature columns:
['WindowID', 'ParticipantID', 'ECGFile', 'StartTime', 'EndTime', 'WakeSleepLabel', 'Mean', 'Std', 'Min', 'Max', 'Range', 'Median', 'IQR', 'RMS', 'Energy']

Missing values:
0

First five feature records:


,WindowID,ParticipantID,ECGFile,StartTime,EndTime,WakeSleepLabel,Mean,Std,Min,Max,Range,Median,IQR,RMS,Energy
0,389,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:01:56.744,2022-06-06 11:02:06.744,1,-0.406652,12.306687,-49.314396,73.840210,123.154602,-1.362580,10.110797,12.313403,379049.75000
1,390,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:01.744,2022-06-06 11:02:11.744,1,0.361844,11.943188,-47.082298,64.899399,111.981697,-0.467454,10.409337,11.948667,356926.59375
2,391,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:06.744,2022-06-06 11:02:16.744,1,-0.220073,12.925522,-47.082298,64.899399,111.981697,-0.651769,11.338026,12.927394,417793.81250
3,392,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:11.744,2022-06-06 11:02:21.744,1,-0.418021,11.949225,-46.175804,56.833618,103.009422,-1.052427,10.612090,11.956534,357396.81250
4,393,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:16.744,2022-06-06 11:02:26.744,1,0.076602,11.623153,-35.881332,61.683338,97.564667,-0.877826,10.615895,11.623405,337758.90625


In [ ]:
# Cell 53 — Optimised ECG feature extraction test on one recording

import time
import numpy as np
import pandas as pd

TEST_RECORDING = recording_summary.iloc[0]

participant_id = TEST_RECORDING["ParticipantID"]
ecg_file = TEST_RECORDING["ECGFile"]

print("Optimised ECG feature extraction test")
print("=" * 60)

print(f"Participant: {participant_id}")
print(f"ECG file:    {ecg_file}")



source_row = recording_times_df[
    (recording_times_df["ParticipantID"] == participant_id)
    &
    (recording_times_df["ECGFile"] == ecg_file)
]

if len(source_row) != 1:
    raise ValueError(
        "Source ECG recording could not be uniquely identified."
    )

source_path = source_row.iloc[0]["Path"]



alignment_row = ecg_alignment_df[
    (ecg_alignment_df["ParticipantID"] == participant_id)
    &
    (ecg_alignment_df["ECGFile"] == ecg_file)
]

if len(alignment_row) != 1:
    raise ValueError(
        "ECG timestamp alignment information not found."
    )

actual_first_timestamp = alignment_row.iloc[0][
    "ActualFirstTimestamp"
]



recording_windows = labelled_windows_df[
    (labelled_windows_df["ParticipantID"] == participant_id)
    &
    (labelled_windows_df["ECGFile"] == ecg_file)
].copy()

print(
    f"Labelled windows: {len(recording_windows):,}"
)

-

start_time = time.time()

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    with zip_ref.open(source_path) as file:

        waveform_df = pd.read_csv(
            file,
            usecols=["EcgWaveform"]
        )

signal_array = pd.to_numeric(
    waveform_df["EcgWaveform"],
    errors="coerce"
).to_numpy(
    dtype=np.float32
)

# Remove invalid samples only if present
valid_signal = np.isfinite(signal_array)

if not valid_signal.all():
    print(
        f"Warning: "
        f"{(~valid_signal).sum():,} invalid waveform samples found."
    )



filtered_signal_array = apply_segment_level_bandpass_filter(
    signal_array
)



SAMPLE_RATE = 250
SAMPLES_PER_WINDOW = 2500

window_start_seconds = (
    recording_windows["StartTime"]
    - actual_first_timestamp
).dt.total_seconds()

start_indices = np.rint(
    window_start_seconds * SAMPLE_RATE
).astype(np.int64)


# Extract features


optimised_results = []

for window, start_index in zip(
    recording_windows.itertuples(index=False),
    start_indices
):

    end_index = (
        start_index
        + SAMPLES_PER_WINDOW
    )

    if start_index < 0:
        continue

    if end_index > len(signal_array):
        continue

    signal_window = filtered_signal_array[
        start_index:end_index
    ]

    if len(signal_window) != SAMPLES_PER_WINDOW:
        continue

    features = extract_time_domain_features(
        signal_window
    )

    if features is None:
        continue

    optimised_results.append({
        "WindowID": window.WindowID,
        "ParticipantID": window.ParticipantID,
        "ECGFile": window.ECGFile,
        "StartTime": window.StartTime,
        "EndTime": window.EndTime,
        "WakeSleepLabel": window.WakeSleepLabel,
        **features
    })

optimised_elapsed = (
    time.time() - start_time
)

optimised_test_df = pd.DataFrame(
    optimised_results
)

print("\nOptimised extraction result")
print("=" * 60)

print(
    f"Waveform samples loaded: "
    f"{len(signal_array):,}"
)

print(
    f"Expected labelled windows: "
    f"{len(recording_windows):,}"
)

print(
    f"Feature rows created: "
    f"{len(optimised_test_df):,}"
)

print(
    f"Processing time: "
    f"{optimised_elapsed:.2f} seconds"
)

print("\nFeature validation:")

if len(optimised_test_df) > 0:

    feature_columns = [
        "Mean",
        "Std",
        "Min",
        "Max",
        "Range",
        "Median",
        "IQR",
        "RMS",
        "Energy"
    ]

    feature_values = optimised_test_df[
        feature_columns
    ].to_numpy(dtype=np.float64)

    print(
        f"Missing feature values: "
        f"{np.isnan(feature_values).sum():,}"
    )

    print(
        f"Infinite feature values: "
        f"{np.isinf(feature_values).sum():,}"
    )

    print(
        f"All features finite: "
        f"{np.isfinite(feature_values).all()}"
    )

    print("\nFirst five optimised feature records:")

    display(
        optimised_test_df.head()
    )

Optimised ECG feature extraction test
Participant: 001
ECG file:    2022_06_06__10_28_26_ECG.csv
Labelled windows: 3,304

Optimised extraction result
Waveform samples loaded: 4,647,250
Expected labelled windows: 3,304
Feature rows created: 3,304
Processing time: 4.33 seconds

Feature validation:
Missing feature values: 0
Infinite feature values: 0
All features finite: True

First five optimised feature records:


,WindowID,ParticipantID,ECGFile,StartTime,EndTime,WakeSleepLabel,Mean,Std,Min,Max,Range,Median,IQR,RMS,Energy
0,389,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:01:56.744,2022-06-06 11:02:06.744,1,-0.406652,12.306687,-49.314396,73.840210,123.154602,-1.362580,10.110797,12.313403,379049.75000
1,390,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:01.744,2022-06-06 11:02:11.744,1,0.361844,11.943188,-47.082298,64.899399,111.981697,-0.467454,10.409337,11.948667,356926.59375
2,391,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:06.744,2022-06-06 11:02:16.744,1,-0.220073,12.925522,-47.082298,64.899399,111.981697,-0.651769,11.338026,12.927394,417793.81250
3,392,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:11.744,2022-06-06 11:02:21.744,1,-0.418021,11.949225,-46.175804,56.833618,103.009422,-1.052427,10.612090,11.956534,357396.81250
4,393,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:16.744,2022-06-06 11:02:26.744,1,0.076602,11.623153,-35.881332,61.683338,97.564667,-0.877826,10.615895,11.623405,337758.90625


In [ ]:
# Cell 54 — Validate optimised features against original extraction

feature_columns = [
    "Mean",
    "Std",
    "Min",
    "Max",
    "Range",
    "Median",
    "IQR",
    "RMS",
    "Energy"
]

# ------------------------------------------------------------
# Compare the original and optimised results
# for the first recording
# ------------------------------------------------------------

original_test = batch_features_df[
    batch_features_df["ECGFile"] == ecg_file
].copy()

optimised_test = optimised_test_df.copy()

# Align both datasets by WindowID
comparison = original_test[
    ["WindowID"] + feature_columns
].merge(
    optimised_test[
        ["WindowID"] + feature_columns
    ],
    on="WindowID",
    suffixes=("_Original", "_Optimised")
)

# Calculate absolute differences
difference_results = {}

for feature in feature_columns:

    difference_results[feature] = np.max(
        np.abs(
            comparison[f"{feature}_Original"]
            -
            comparison[f"{feature}_Optimised"]
        )
    )

comparison_summary = pd.DataFrame(
    {
        "Feature": list(difference_results.keys()),
        "MaximumAbsoluteDifference": list(
            difference_results.values()
        )
    }
)

print("Optimised feature extraction validation")
print("=" * 60)

print(
    f"Original feature rows: "
    f"{len(original_test):,}"
)

print(
    f"Optimised feature rows: "
    f"{len(optimised_test):,}"
)

print(
    f"Matched windows: "
    f"{len(comparison):,}"
)

print("\nMaximum absolute difference by feature:")

display(
    comparison_summary
)

# Overall validation
max_difference = (
    comparison_summary[
        "MaximumAbsoluteDifference"
    ].max()
)

print(
    f"\nMaximum difference across all features: "
    f"{max_difference:.12f}"
)

if max_difference < 1e-5:
    print(
        "\nValidation result: PASSED"
    )
    print(
        "Optimised extraction reproduces the "
        "original feature values."
    )
else:
    print(
        "\nValidation result: REVIEW REQUIRED"
    )
    print(
        "Differences were detected between "
        "the two extraction methods."
    )

Optimised feature extraction validation
Original feature rows: 3,304
Optimised feature rows: 3,304
Matched windows: 3,304

Maximum absolute difference by feature:


,Feature,MaximumAbsoluteDifference
0,Mean,0.0
1,Std,0.0
2,Min,0.0
3,Max,0.0
4,Range,0.0
5,Median,0.0
6,IQR,0.0
7,RMS,0.0
8,Energy,0.0



Maximum difference across all features: 0.000000000000

Validation result: PASSED
Optimised extraction reproduces the original feature values.


In [ ]:
# Cell 55 — Full optimised ECG feature extraction

import time
import numpy as np
import pandas as pd

SAMPLE_RATE = 250
SAMPLES_PER_WINDOW = 2500

all_feature_results = []

total_start_time = time.time()

print("Full optimised ECG feature extraction")
print("=" * 60)
print(f"Total recordings: {len(recording_summary):,}")
print(f"Total labelled windows: {len(labelled_windows_df):,}")

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    for recording_number, (_, recording) in enumerate(
        recording_summary.iterrows(),
        start=1
    ):

        participant_id = recording["ParticipantID"]
        ecg_file = recording["ECGFile"]

        # Source recording path
        source_row = recording_times_df[
            (recording_times_df["ParticipantID"] == participant_id)
            &
            (recording_times_df["ECGFile"] == ecg_file)
        ]

        if len(source_row) != 1:
            print(
                f"Skipping recording {recording_number}: "
                f"source path not uniquely identified."
            )
            continue

        source_path = source_row.iloc[0]["Path"]

        # Actual first ECG timestamp
        alignment_row = ecg_alignment_df[
            (ecg_alignment_df["ParticipantID"] == participant_id)
            &
            (ecg_alignment_df["ECGFile"] == ecg_file)
        ]

        if len(alignment_row) != 1:
            print(
                f"Skipping recording {recording_number}: "
                f"timestamp alignment not found."
            )
            continue

        actual_first_timestamp = alignment_row.iloc[0][
            "ActualFirstTimestamp"
        ]

        # Windows belonging to this recording
        recording_windows = labelled_windows_df[
            (labelled_windows_df["ParticipantID"] == participant_id)
            &
            (labelled_windows_df["ECGFile"] == ecg_file)
        ].copy()

        if len(recording_windows) == 0:
            continue

        print(
            f"Processing {recording_number:02d}/"
            f"{len(recording_summary)} | "
            f"Participant {participant_id} | "
            f"{ecg_file} | "
            f"{len(recording_windows):,} windows"
        )



        with zip_ref.open(source_path) as file:

            waveform_df = pd.read_csv(
                file,
                usecols=["EcgWaveform"]
            )

        signal_array = pd.to_numeric(
            waveform_df["EcgWaveform"],
            errors="coerce"
        ).to_numpy(
            dtype=np.float32
        )

-

        filtered_signal_array = apply_segment_level_bandpass_filter(
            signal_array
        )


        window_start_seconds = (
            recording_windows["StartTime"]
            - actual_first_timestamp
        ).dt.total_seconds()

        start_indices = np.rint(
            window_start_seconds * SAMPLE_RATE
        ).astype(np.int64)



        for window, start_index in zip(
            recording_windows.itertuples(index=False),
            start_indices
        ):

            end_index = (
                start_index
                + SAMPLES_PER_WINDOW
            )

            if start_index < 0:
                continue

            if end_index > len(signal_array):
                continue

            signal_window = filtered_signal_array[
                start_index:end_index
            ]

            if len(signal_window) != SAMPLES_PER_WINDOW:
                continue

            features = extract_time_domain_features(
                signal_window
            )

            if features is None:
                continue

            all_feature_results.append({
                "WindowID": window.WindowID,
                "ParticipantID": window.ParticipantID,
                "ECGFile": window.ECGFile,
                "StartTime": window.StartTime,
                "EndTime": window.EndTime,
                "WakeSleepLabel": window.WakeSleepLabel,
                **features
            })

# ------------------------------------------------------------
# Create final feature dataframe
# ------------------------------------------------------------

ecg_features_df = pd.DataFrame(
    all_feature_results
)

total_elapsed = (
    time.time() - total_start_time
)

print("\nFull extraction completed")
print("=" * 60)

print(
    f"Feature rows created: "
    f"{len(ecg_features_df):,}"
)

print(
    f"Expected labelled windows: "
    f"{len(labelled_windows_df):,}"
)

print(
    f"Processing time: "
    f"{total_elapsed:.2f} seconds"
)

print(
    f"Processing time: "
    f"{total_elapsed / 60:.2f} minutes"
)

print("\nFeature columns:")

print(
    ecg_features_df.columns.tolist()
)

print("\nMissing values:")

print(
    ecg_features_df.isna().sum().sum()
)

print("\nFirst five feature records:")

display(
    ecg_features_df.head()
)

Full optimised ECG feature extraction
Total recordings: 34
Total labelled windows: 421,175
Processing 01/34 | Participant 001 | 2022_06_06__10_28_26_ECG.csv | 3,304 windows
Processing 02/34 | Participant 001 | 2022_06_06__15_38_43_ECG.csv | 9,591 windows
Processing 03/34 | Participant 001 | 2022_06_07__10_42_46_ECG.csv | 478 windows
Processing 04/34 | Participant 002 | 2022_06_21__08_03_26_ECG.csv | 459 windows
Processing 05/34 | Participant 002 | 2022_06_21__08_43_49_ECG.csv | 2,258 windows
Processing 06/34 | Participant 002 | 2022_06_21__11_53_12_ECG.csv | 12,083 windows
Processing 07/34 | Participant 003 | 2022_08_01__10_08_04_ECG.csv | 14,601 windows
Processing 08/34 | Participant 004 | 2022_08_02__13_54_45_ECG.csv | 14,323 windows
Processing 09/34 | Participant 005 | 2022_08_03__12_17_03_ECG.csv | 14,170 windows
Processing 10/34 | Participant 006 | 2022_08_04__09_14_28_ECG.csv | 14,324 windows
Processing 11/34 | Participant 007 | 2023_03_15__10_38_56_ECG.csv | 14,283 windows
Proce

,WindowID,ParticipantID,ECGFile,StartTime,EndTime,WakeSleepLabel,Mean,Std,Min,Max,Range,Median,IQR,RMS,Energy
0,389,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:01:56.744,2022-06-06 11:02:06.744,1,-0.406652,12.306687,-49.314396,73.840210,123.154602,-1.362580,10.110797,12.313403,379049.75000
1,390,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:01.744,2022-06-06 11:02:11.744,1,0.361844,11.943188,-47.082298,64.899399,111.981697,-0.467454,10.409337,11.948667,356926.59375
2,391,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:06.744,2022-06-06 11:02:16.744,1,-0.220073,12.925522,-47.082298,64.899399,111.981697,-0.651769,11.338026,12.927394,417793.81250
3,392,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:11.744,2022-06-06 11:02:21.744,1,-0.418021,11.949225,-46.175804,56.833618,103.009422,-1.052427,10.612090,11.956534,357396.81250
4,393,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:16.744,2022-06-06 11:02:26.744,1,0.076602,11.623153,-35.881332,61.683338,97.564667,-0.877826,10.615895,11.623405,337758.90625


In [ ]:
# Cell 56 — Diagnose missing ECG feature windows

expected_window_ids = set(
    labelled_windows_df["WindowID"]
)

extracted_window_ids = set(
    ecg_features_df["WindowID"]
)

missing_window_ids = sorted(
    expected_window_ids - extracted_window_ids
)

missing_windows_df = labelled_windows_df[
    labelled_windows_df["WindowID"].isin(
        missing_window_ids
    )
].copy()

print("ECG feature extraction completeness check")
print("=" * 60)

print(
    f"Expected labelled windows: "
    f"{len(expected_window_ids):,}"
)

print(
    f"Extracted feature windows: "
    f"{len(extracted_window_ids):,}"
)

print(
    f"Missing feature windows: "
    f"{len(missing_window_ids):,}"
)

print("\nMissing windows by participant:")

display(
    missing_windows_df[
        [
            "WindowID",
            "ParticipantID",
            "ECGFile",
            "StartTime",
            "EndTime",
            "WakeSleepLabel"
        ]
    ]
)

print("\nMissing windows per participant:")

display(
    missing_windows_df[
        "ParticipantID"
    ].value_counts()
    .sort_index()
    .rename("MissingWindows")
    .reset_index()
)

ECG feature extraction completeness check
Expected labelled windows: 421,175
Extracted feature windows: 421,166
Missing feature windows: 9

Missing windows by participant:


,WindowID,ParticipantID,ECGFile,StartTime,EndTime,WakeSleepLabel
3693,3693,001,2022_06_06__15_38_43_ECG.csv,2022-06-06 15:38:43,2022-06-06 15:38:53,1
18206,18206,002,2022_06_21__08_03_26_ECG.csv,2022-06-21 08:03:26,2022-06-21 08:03:36,1
125452,125452,008,2023_03_28__11_36_58_ECG.csv,2023-03-28 11:36:58,2023-03-28 11:37:08,1
142265,142265,009,2023_04_05__10_41_32_ECG.csv,2023-04-05 10:41:32,2023-04-05 10:41:42,1
300833,300833,018,2023_05_16__16_10_03_ECG.csv,2023-05-16 16:10:03,2023-05-16 16:10:13,1
319220,319220,019,2023_07_18__15_22_49_ECG.csv,2023-07-18 15:22:49,2023-07-18 15:22:59,1
336783,336783,020,2023_08_01__10_55_09_ECG.csv,2023-08-01 10:55:09,2023-08-01 10:55:19,1
387561,390790,023,2023_08_09__10_21_22_ECG.csv,2023-08-09 10:21:22,2023-08-09 10:21:32,1
443050,443022,026,2023_08_17__10_41_11_ECG.csv,2023-08-17 10:41:11,2023-08-17 10:41:21,1



Missing windows per participant:


,ParticipantID,MissingWindows
0,001,1
1,002,1
2,008,1
3,009,1
4,018,1
5,019,1
6,020,1
7,023,1
8,026,1


In [ ]:
# Cell 57 — Validate boundary-related missing ECG windows

print("Validation of missing ECG feature windows")
print("=" * 60)

missing_diagnostics = missing_windows_df.copy()



missing_diagnostics = missing_diagnostics.merge(
    recording_times_df[
        [
            "ParticipantID",
            "ECGFile",
            "ECGStart",
            "ECGEnd"
        ]
    ],
    on=["ParticipantID", "ECGFile"],
    how="left"
)



missing_diagnostics["SecondsFromRecordingStart"] = (
    missing_diagnostics["StartTime"]
    - missing_diagnostics["ECGStart"]
).dt.total_seconds()



missing_diagnostics["SecondsBeforeActualFirstSample"] = (
    missing_diagnostics["StartTime"]
    - missing_diagnostics["StartTime"]
)

# Use the actual timestamp from alignment metadata
missing_diagnostics = missing_diagnostics.merge(
    ecg_alignment_df[
        [
            "ParticipantID",
            "ECGFile",
            "ActualFirstTimestamp",
            "OffsetSeconds"
        ]
    ],
    on=["ParticipantID", "ECGFile"],
    how="left"
)

missing_diagnostics["WindowStartToActualSample"] = (
    missing_diagnostics["StartTime"]
    - missing_diagnostics["ActualFirstTimestamp"]
).dt.total_seconds()

print(
    f"Missing windows examined: "
    f"{len(missing_diagnostics):,}"
)

print("\nTiming relationship:")

display(
    missing_diagnostics[
        [
            "ParticipantID",
            "ECGFile",
            "StartTime",
            "ECGStart",
            "ActualFirstTimestamp",
            "OffsetSeconds",
            "SecondsFromRecordingStart",
            "WindowStartToActualSample"
        ]
    ]
)

print("\nSummary:")

print(
    "Missing windows at reconstructed recording start: "
    f"{(
        missing_diagnostics["SecondsFromRecordingStart"] == 0
    ).sum():,}"
)

print(
    "Missing windows starting before first actual ECG sample: "
    f"{(
        missing_diagnostics["WindowStartToActualSample"] < 0
    ).sum():,}"
)

print(
    "\nMean recording timestamp offset: "
    f"{missing_diagnostics['OffsetSeconds'].mean():.4f} seconds"
)

Validation of missing ECG feature windows
Missing windows examined: 9

Timing relationship:


,ParticipantID,ECGFile,StartTime,ECGStart,ActualFirstTimestamp,OffsetSeconds,SecondsFromRecordingStart,WindowStartToActualSample
0,001,2022_06_06__15_38_43_ECG.csv,2022-06-06 15:38:43,2022-06-06 15:38:43,2022-06-06 15:38:43.457,0.457,0.0,-0.457
1,002,2022_06_21__08_03_26_ECG.csv,2022-06-21 08:03:26,2022-06-21 08:03:26,2022-06-21 08:03:26.421,0.421,0.0,-0.421
2,008,2023_03_28__11_36_58_ECG.csv,2023-03-28 11:36:58,2023-03-28 11:36:58,2023-03-28 11:36:58.446,0.446,0.0,-0.446
3,009,2023_04_05__10_41_32_ECG.csv,2023-04-05 10:41:32,2023-04-05 10:41:32,2023-04-05 10:41:32.454,0.454,0.0,-0.454
4,018,2023_05_16__16_10_03_ECG.csv,2023-05-16 16:10:03,2023-05-16 16:10:03,2023-05-16 16:10:03.459,0.459,0.0,-0.459
5,019,2023_07_18__15_22_49_ECG.csv,2023-07-18 15:22:49,2023-07-18 15:22:49,2023-07-18 15:22:49.428,0.428,0.0,-0.428
6,020,2023_08_01__10_55_09_ECG.csv,2023-08-01 10:55:09,2023-08-01 10:55:09,2023-08-01 10:55:09.444,0.444,0.0,-0.444
7,023,2023_08_09__10_21_22_ECG.csv,2023-08-09 10:21:22,2023-08-09 10:21:22,2023-08-09 10:21:22.428,0.428,0.0,-0.428
8,026,2023_08_17__10_41_11_ECG.csv,2023-08-17 10:41:11,2023-08-17 10:41:11,2023-08-17 10:41:11.427,0.427,0.0,-0.427



Summary:
Missing windows at reconstructed recording start: 9
Missing windows starting before first actual ECG sample: 9

Mean recording timestamp offset: 0.4404 seconds


In [ ]:
# Cell 58 — Finalise ECG feature dataset after boundary validation

expected_windows = len(labelled_windows_df)
extracted_windows = len(ecg_features_df)
excluded_windows = expected_windows - extracted_windows

print("Final ECG feature dataset")
print("=" * 60)

print(
    f"Labelled ECG windows:        {expected_windows:,}"
)

print(
    f"Complete feature windows:    {extracted_windows:,}"
)

print(
    f"Boundary windows excluded:   {excluded_windows:,}"
)

print(
    f"Feature coverage:            "
    f"{extracted_windows / expected_windows * 100:.4f}%"
)



excluded_ids = set(
    labelled_windows_df["WindowID"]
) - set(
    ecg_features_df["WindowID"]
)

diagnosed_ids = set(
    missing_diagnostics["WindowID"]
)

print("\nExclusion validation:")
print(
    f"Excluded IDs match diagnosed boundary IDs: "
    f"{excluded_ids == diagnosed_ids}"
)



feature_columns = [
    "Mean",
    "Std",
    "Min",
    "Max",
    "Range",
    "Median",
    "IQR",
    "RMS",
    "Energy"
]

feature_values = ecg_features_df[
    feature_columns
].to_numpy(
    dtype=np.float64
)

print(
    f"Missing feature values:      "
    f"{np.isnan(feature_values).sum():,}"
)

print(
    f"Infinite feature values:     "
    f"{np.isinf(feature_values).sum():,}"
)

print(
    f"Unique WindowIDs:             "
    f"{ecg_features_df['WindowID'].nunique():,}"
)

print(
    f"Unique participants:          "
    f"{ecg_features_df['ParticipantID'].nunique():,}"
)

print("\nFeature extraction status:")

if (
    excluded_ids == diagnosed_ids
    and np.isfinite(feature_values).all()
    and ecg_features_df["WindowID"].is_unique
):
    print("PASSED")
    print(
        "ECG feature dataset is ready for "
        "Section 9 validation."
    )
else:
    print("REVIEW REQUIRED")

Final ECG feature dataset
Labelled ECG windows:        421,175
Complete feature windows:    421,166
Boundary windows excluded:   9
Feature coverage:            99.9979%

Exclusion validation:
Excluded IDs match diagnosed boundary IDs: True
Missing feature values:      0
Infinite feature values:     0
Unique WindowIDs:             421,166
Unique participants:          30

Feature extraction status:
PASSED
ECG feature dataset is ready for Section 9 validation.


## 9. Feature Validation

The extracted ECG features are now validated before being used for machine-learning and deep-learning dataset preparation.

Validation focuses on numerical integrity, feature variability, participant coverage, and consistency of the extracted feature dataset. This step ensures that the feature representation is suitable for subsequent modelling and that no invalid or degenerate feature values have been introduced during extraction.

The validation is intentionally limited to checks that provide useful evidence for the modelling pipeline, avoiding unnecessary additional analysis.

In [ ]:
# Cell 59 — ECG feature validation summary

feature_columns = [
    "Mean",
    "Std",
    "Min",
    "Max",
    "Range",
    "Median",
    "IQR",
    "RMS",
    "Energy"
]

print("ECG feature validation")
print("=" * 60)

print(
    f"Feature dataset rows: "
    f"{len(ecg_features_df):,}"
)

print(
    f"Feature columns: "
    f"{len(feature_columns)}"
)

print(
    f"Participants represented: "
    f"{ecg_features_df['ParticipantID'].nunique():,}"
)

print(
    f"Unique ECG recordings: "
    f"{ecg_features_df['ECGFile'].nunique():,}"
)

print("\nFeature statistics:")
print("=" * 60)

feature_summary = ecg_features_df[
    feature_columns
].describe().T

feature_summary = feature_summary[
    [
        "mean",
        "std",
        "min",
        "25%",
        "50%",
        "75%",
        "max"
    ]
]

display(
    feature_summary
)

print("\nNumerical validity checks:")
print("=" * 60)

feature_array = ecg_features_df[
    feature_columns
].to_numpy(
    dtype=np.float64
)

print(
    f"Missing values: "
    f"{np.isnan(feature_array).sum():,}"
)

print(
    f"Infinite values: "
    f"{np.isinf(feature_array).sum():,}"
)

print(
    f"Constant features: "
    f"{
        (
            ecg_features_df[feature_columns].nunique() <= 1
        ).sum()
    }"
)

print(
    f"Negative standard deviations: "
    f"{
        (
            ecg_features_df["Std"] < 0
        ).sum()
    }"
)

print(
    f"Negative IQR values: "
    f"{
        (
            ecg_features_df["IQR"] < 0
        ).sum()
    }"
)

print(
    f"Negative RMS values: "
    f"{
        (
            ecg_features_df["RMS"] < 0
        ).sum()
    }"
)

print(
    f"Negative Energy values: "
    f"{
        (
            ecg_features_df["Energy"] < 0
        ).sum()
    }"
)

ECG feature validation
Feature dataset rows: 421,166
Feature columns: 9
Participants represented: 30
Unique ECG recordings: 34

Feature statistics:


,mean,std,min,25%,50%,75%,max
Mean,1.000962e-02,4.228768e+00,-80.897621,-0.143467,0.000211,1.438284e-01,8.875725e+01
Std,7.762782e+01,2.074931e+02,0.859716,12.005438,18.386650,3.677956e+01,1.466480e+03
Min,-2.305620e+02,5.039342e+02,-4937.534180,-148.228439,-66.911888,-3.608340e+01,-2.174848e+00
Max,2.280235e+02,4.931148e+02,2.145850,47.980694,72.501720,1.138066e+02,4.894184e+03
Range,4.585855e+02,9.891414e+02,4.410201,92.381395,142.789726,2.732919e+02,7.449640e+03
Median,-1.220816e+01,7.638608e+01,-684.707275,-6.402672,-3.806694,-1.899681e+00,6.480886e+02
IQR,1.036591e+02,3.572053e+02,1.198436,9.607294,14.906033,2.743159e+01,2.635926e+03
RMS,7.764172e+01,2.075316e+02,0.859729,12.006318,18.388742,3.678218e+01,1.466738e+03
Energy,1.227278e+08,5.159174e+08,1847.833252,360379.179688,845364.593750,3.382185e+06,5.378304e+09



Numerical validity checks:
Missing values: 0
Infinite values: 0
Constant features: 0
Negative standard deviations: 0
Negative IQR values: 0
Negative RMS values: 0
Negative Energy values: 0


## 10. Machine-Learning Dataset Preparation

The validated ECG feature dataset is now prepared for traditional machine-learning models.

The machine-learning pathway will use the engineered ECG features together with the validated Wake/Sleep labels. The dataset will retain participant identifiers and relevant metadata for traceability, while the modelling feature matrix will contain only the numerical ECG features.

At this stage, no train/validation/test split is performed. Dataset splitting and participant-level separation will be handled in the modelling notebook to prevent information leakage between participants.

The prepared dataset will be checked for feature completeness, label availability, class representation, and participant coverage before being passed to the modelling stage.

In [ ]:
# Cell 61 — Prepare machine-learning feature dataset

ml_feature_columns = [
    "Mean",
    "Std",
    "Min",
    "Max",
    "Range",
    "Median",
    "IQR",
    "RMS",
    "Energy"
]



ml_dataset_df = ecg_features_df[
    [
        "WindowID",
        "ParticipantID",
        "ECGFile",
        "StartTime",
        "EndTime",
        "WakeSleepLabel",
        *ml_feature_columns
    ]
].copy()



print("Machine-learning dataset preparation")
print("=" * 60)

print(
    f"ML dataset rows: "
    f"{len(ml_dataset_df):,}"
)

print(
    f"ML features: "
    f"{len(ml_feature_columns)}"
)

print(
    f"Participants: "
    f"{ml_dataset_df['ParticipantID'].nunique():,}"
)

print(
    f"ECG recordings: "
    f"{ml_dataset_df['ECGFile'].nunique():,}"
)

print("\nClass distribution:")

display(
    ml_dataset_df[
        "WakeSleepLabel"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("WakeSleepLabel")
    .reset_index(name="Count")
)



print("\nFeature completeness:")

print(
    f"Missing feature values: "
    f"{ml_dataset_df[ml_feature_columns].isna().sum().sum():,}"
)

print(
    f"Infinite feature values: "
    f"{np.isinf(
        ml_dataset_df[ml_feature_columns]
        .to_numpy(dtype=np.float64)
    ).sum():,}"
)

print(
    f"Missing labels: "
    f"{ml_dataset_df['WakeSleepLabel'].isna().sum():,}"
)

print(
    f"Unique WindowIDs: "
    f"{ml_dataset_df['WindowID'].nunique():,}"
)

print("\nFirst five ML records:")

display(
    ml_dataset_df.head()
)

Machine-learning dataset preparation
ML dataset rows: 421,166
ML features: 9
Participants: 30
ECG recordings: 34

Class distribution:


,WakeSleepLabel,Count
0,0,87407
1,1,333759



Feature completeness:
Missing feature values: 0
Infinite feature values: 0
Missing labels: 0
Unique WindowIDs: 421,166

First five ML records:


,WindowID,ParticipantID,ECGFile,StartTime,EndTime,WakeSleepLabel,Mean,Std,Min,Max,Range,Median,IQR,RMS,Energy
0,389,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:01:56.744,2022-06-06 11:02:06.744,1,-0.406652,12.306687,-49.314396,73.840210,123.154602,-1.362580,10.110797,12.313403,379049.75000
1,390,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:01.744,2022-06-06 11:02:11.744,1,0.361844,11.943188,-47.082298,64.899399,111.981697,-0.467454,10.409337,11.948667,356926.59375
2,391,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:06.744,2022-06-06 11:02:16.744,1,-0.220073,12.925522,-47.082298,64.899399,111.981697,-0.651769,11.338026,12.927394,417793.81250
3,392,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:11.744,2022-06-06 11:02:21.744,1,-0.418021,11.949225,-46.175804,56.833618,103.009422,-1.052427,10.612090,11.956534,357396.81250
4,393,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:16.744,2022-06-06 11:02:26.744,1,0.076602,11.623153,-35.881332,61.683338,97.564667,-0.877826,10.615895,11.623405,337758.90625


## 11. Deep-Learning Dataset Preparation

The validated ECG windows are now prepared for the deep-learning pathway.

Unlike the traditional machine-learning dataset, the deep-learning representation will preserve the sequential ECG waveform information within each 10-second window. Each window contains 2,500 samples corresponding to the validated 250 Hz sampling frequency.

The Wake/Sleep label and participant identifier will be retained alongside each ECG window to support supervised learning and participant-level dataset separation during modelling.

Because the complete waveform dataset is large, the deep-learning preparation will use an efficient representation and avoid unnecessary duplication of the raw ECG archive in memory.

> **Methodological limitation:** The deep-learning pathway currently uses the original ECG waveform with per-window standardisation, whereas the engineered ML features (Section 8) are calculated from the validated band-pass filtered signal. Consequently, some residual baseline variation may remain in the DL input.


In [ ]:
# Cell 63 — Prepare deep-learning window metadata

DL_SAMPLE_RATE = 250
DL_SAMPLES_PER_WINDOW = 2500



dl_metadata_df = labelled_windows_df[
    [
        "WindowID",
        "ParticipantID",
        "ECGFile",
        "StartTime",
        "EndTime",
        "WakeSleepLabel"
    ]
].copy()



dl_metadata_df = dl_metadata_df[
    dl_metadata_df["WindowID"].isin(
        ecg_features_df["WindowID"]
    )
].copy()



dl_metadata_df = dl_metadata_df.merge(
    recording_times_df[
        [
            "ParticipantID",
            "ECGFile",
            "Path"
        ]
    ],
    on=["ParticipantID", "ECGFile"],
    how="left"
)


# Add actual ECG sample indexing information


dl_metadata_df = dl_metadata_df.merge(
    ecg_alignment_df[
        [
            "ParticipantID",
            "ECGFile",
            "ActualFirstTimestamp"
        ]
    ],
    on=["ParticipantID", "ECGFile"],
    how="left"
)

dl_metadata_df["StartSample"] = np.rint(
    (
        dl_metadata_df["StartTime"]
        - dl_metadata_df["ActualFirstTimestamp"]
    ).dt.total_seconds()
    * DL_SAMPLE_RATE
).astype(np.int64)

dl_metadata_df["EndSample"] = (
    dl_metadata_df["StartSample"]
    + DL_SAMPLES_PER_WINDOW
)


# Validate metadata


print("Deep-learning dataset metadata")
print("=" * 60)

print(
    f"DL windows: "
    f"{len(dl_metadata_df):,}"
)

print(
    f"Participants: "
    f"{dl_metadata_df['ParticipantID'].nunique():,}"
)

print(
    f"ECG recordings: "
    f"{dl_metadata_df['ECGFile'].nunique():,}"
)

print(
    f"Samples per window: "
    f"{DL_SAMPLES_PER_WINDOW:,}"
)

print(
    f"Sampling frequency: "
    f"{DL_SAMPLE_RATE} Hz"
)

print("\nLabel distribution:")

display(
    dl_metadata_df[
        "WakeSleepLabel"
    ]
    .value_counts()
    .sort_index()
    .rename_axis("WakeSleepLabel")
    .reset_index(name="Count")
)

print("\nMetadata validation:")

print(
    f"Missing source paths: "
    f"{dl_metadata_df['Path'].isna().sum():,}"
)

print(
    f"Missing alignment timestamps: "
    f"{dl_metadata_df['ActualFirstTimestamp'].isna().sum():,}"
)

print(
    f"Missing labels: "
    f"{dl_metadata_df['WakeSleepLabel'].isna().sum():,}"
)

print(
    f"Invalid sample indices: "
    f"{
        (
            dl_metadata_df["StartSample"] < 0
        ).sum()
    }"
)

print(
    f"Incorrect window lengths: "
    f"{
        (
            (
                dl_metadata_df["EndSample"]
                -
                dl_metadata_df["StartSample"]
            )
            != DL_SAMPLES_PER_WINDOW
        ).sum()
    }"
)

print("\nFirst five DL metadata records:")

display(
    dl_metadata_df.head()
)

Deep-learning dataset metadata
DL windows: 421,166
Participants: 30
ECG recordings: 34
Samples per window: 2,500
Sampling frequency: 250 Hz

Label distribution:


,WakeSleepLabel,Count
0,0,87407
1,1,333759



Metadata validation:
Missing source paths: 0
Missing alignment timestamps: 0
Missing labels: 0
Invalid sample indices: 0
Incorrect window lengths: 0

First five DL metadata records:


,WindowID,ParticipantID,ECGFile,StartTime,EndTime,WakeSleepLabel,Path,ActualFirstTimestamp,StartSample,EndSample
0,389,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:01:56.744,2022-06-06 11:02:06.744,1,Per_Participant_Sensor_Data/001/001_Zephyr/202...,2022-06-06 10:28:26.448,502574,505074
1,390,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:01.744,2022-06-06 11:02:11.744,1,Per_Participant_Sensor_Data/001/001_Zephyr/202...,2022-06-06 10:28:26.448,503824,506324
2,391,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:06.744,2022-06-06 11:02:16.744,1,Per_Participant_Sensor_Data/001/001_Zephyr/202...,2022-06-06 10:28:26.448,505074,507574
3,392,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:11.744,2022-06-06 11:02:21.744,1,Per_Participant_Sensor_Data/001/001_Zephyr/202...,2022-06-06 10:28:26.448,506324,508824
4,393,001,2022_06_06__10_28_26_ECG.csv,2022-06-06 11:02:16.744,2022-06-06 11:02:26.744,1,Per_Participant_Sensor_Data/001/001_Zephyr/202...,2022-06-06 10:28:26.448,507574,510074


In [ ]:
# Cell 64 — Test single-window deep-learning ECG extraction

import csv
import numpy as np
import zipfile
from itertools import islice

print("Testing single-window DL ECG extraction")
print("=" * 60)


# Select first validated DL window


test_row = dl_metadata_df.iloc[0]

participant_id = test_row["ParticipantID"]
ecg_file = test_row["ECGFile"]
zip_internal_path = test_row["Path"]

start_sample = int(test_row["StartSample"])
end_sample = int(test_row["EndSample"])

print(f"Participant: {participant_id}")
print(f"ECG file: {ecg_file}")
print(f"Start sample: {start_sample:,}")
print(f"End sample: {end_sample:,}")
print(f"Expected samples: {DL_SAMPLES_PER_WINDOW:,}")


# Extract required waveform samples


with zipfile.ZipFile(zip_path, "r") as zip_ref:

    with zip_ref.open(zip_internal_path) as file:

        reader = csv.DictReader(
            line.decode("utf-8")
            for line in file
        )

        selected_rows = islice(
            reader,
            start_sample,
            end_sample
        )

        waveform = np.array(
            [
                float(row["EcgWaveform"])
                for row in selected_rows
            ],
            dtype=np.float32
        )


# Validate extraction


print("\nExtraction result")
print("-" * 60)

print(
    f"Samples extracted: "
    f"{len(waveform):,}"
)

print(
    f"Expected samples: "
    f"{DL_SAMPLES_PER_WINDOW:,}"
)

print(
    f"Missing samples: "
    f"{DL_SAMPLES_PER_WINDOW - len(waveform):,}"
)

# Standardise waveform


waveform_mean = np.mean(waveform)
waveform_std = np.std(waveform)

if waveform_std > 0:
    waveform_standardised = (
        waveform - waveform_mean
    ) / waveform_std
else:
    waveform_standardised = waveform.copy()

print("\nWaveform statistics")
print("-" * 60)

print(f"Raw mean: {waveform_mean:.4f}")
print(f"Raw standard deviation: {waveform_std:.4f}")

print(
    f"Standardised mean: "
    f"{np.mean(waveform_standardised):.6f}"
)

print(
    f"Standardised standard deviation: "
    f"{np.std(waveform_standardised):.6f}"
)

print(
    f"Standardised minimum: "
    f"{np.min(waveform_standardised):.4f}"
)

print(
    f"Standardised maximum: "
    f"{np.max(waveform_standardised):.4f}"
)


# Final validation


test_passed = (
    len(waveform) == DL_SAMPLES_PER_WINDOW
    and np.isfinite(waveform).all()
    and np.isfinite(waveform_standardised).all()
    and np.isclose(
        np.mean(waveform_standardised),
        0,
        atol=1e-5
    )
    and np.isclose(
        np.std(waveform_standardised),
        1,
        atol=1e-5
    )
)

print("\nDL single-window validation:")
print("PASSED" if test_passed else "FAILED")

Testing single-window DL ECG extraction
Participant: 001
ECG file: 2022_06_06__10_28_26_ECG.csv
Start sample: 502,574
End sample: 505,074
Expected samples: 2,500

Extraction result
------------------------------------------------------------
Samples extracted: 2,500
Expected samples: 2,500
Missing samples: 0

Waveform statistics
------------------------------------------------------------
Raw mean: 2040.8544
Raw standard deviation: 15.5405
Standardised mean: 0.000002
Standardised standard deviation: 1.000000
Standardised minimum: -4.6880
Standardised maximum: 4.8998

DL single-window validation:
PASSED


In [ ]:
# Cell 65 — Multi-recording DL waveform extraction validation

import csv
import numpy as np
import zipfile
from itertools import islice

print("Multi-recording DL waveform validation")
print("=" * 60)


# Select up to 3 windows from different ECG recordings


test_windows = (
    dl_metadata_df
    .drop_duplicates(
        subset=["ParticipantID", "ECGFile"]
    )
    .head(3)
    .copy()
)

validation_results = []

with zipfile.ZipFile(zip_path, "r") as zip_ref:

    for _, row in test_windows.iterrows():

        participant_id = row["ParticipantID"]
        ecg_file = row["ECGFile"]
        source_path = row["Path"]

        start_sample = int(row["StartSample"])
        end_sample = int(row["EndSample"])

        with zip_ref.open(source_path) as file:

            reader = csv.DictReader(
                line.decode("utf-8")
                for line in file
            )

            selected_rows = islice(
                reader,
                start_sample,
                end_sample
            )

            waveform = np.array(
                [
                    float(item["EcgWaveform"])
                    for item in selected_rows
                ],
                dtype=np.float32
            )

        # Standardise
        waveform_std = np.std(waveform)

        if waveform_std > 0:
            standardised = (
                waveform - np.mean(waveform)
            ) / waveform_std
        else:
            standardised = waveform.copy()

        validation_results.append({
            "ParticipantID": participant_id,
            "ECGFile": ecg_file,
            "Samples": len(waveform),
            "ExpectedSamples": DL_SAMPLES_PER_WINDOW,
            "MissingSamples": (
                DL_SAMPLES_PER_WINDOW - len(waveform)
            ),
            "FiniteRaw": np.isfinite(waveform).all(),
            "StandardisedMean": np.mean(standardised),
            "StandardisedStd": np.std(standardised),
            "FiniteStandardised": np.isfinite(
                standardised
            ).all()
        })

dl_validation_df = pd.DataFrame(
    validation_results
)

print("Validation results:")
print()

display(dl_validation_df)


# Overall validation


all_correct_length = (
    dl_validation_df["Samples"]
    == DL_SAMPLES_PER_WINDOW
).all()

all_complete = (
    dl_validation_df["MissingSamples"]
    == 0
).all()

all_finite = (
    dl_validation_df["FiniteRaw"]
    &
    dl_validation_df["FiniteStandardised"]
).all()

all_standardised = (
    np.isclose(
        dl_validation_df["StandardisedMean"],
        0,
        atol=1e-5
    )
    &
    np.isclose(
        dl_validation_df["StandardisedStd"],
        1,
        atol=1e-5
    )
).all()

print("\nOverall DL validation:")
print("=" * 60)

print(
    f"Correct window length: {all_correct_length}"
)

print(
    f"No missing samples: {all_complete}"
)

print(
    f"All values finite: {all_finite}"
)

print(
    f"Standardisation valid: {all_standardised}"
)

dl_validation_passed = (
    all_correct_length
    and all_complete
    and all_finite
    and all_standardised
)

print(
    "\nValidation result: "
    + ("PASSED" if dl_validation_passed else "FAILED")
)

Multi-recording DL waveform validation
Validation results:



,ParticipantID,ECGFile,Samples,ExpectedSamples,MissingSamples,FiniteRaw,StandardisedMean,StandardisedStd,FiniteStandardised
0,001,2022_06_06__10_28_26_ECG.csv,2500,2500,0,True,1.931763e-06,1.0,True
1,001,2022_06_06__15_38_43_ECG.csv,2500,2500,0,True,1.708984e-06,1.0,True
2,001,2022_06_07__10_42_46_ECG.csv,2500,2500,0,True,1.129150e-07,1.0,True



Overall DL validation:
Correct window length: True
No missing samples: True
All values finite: True
Standardisation valid: True

Validation result: PASSED


## 12. Final Dataset Validation

The machine-learning and deep-learning dataset preparations are now validated together to ensure that both modelling pathways are based on the same set of reliable ECG windows.

Final validation checks participant coverage, window counts, Wake/Sleep class representation, unique window identifiers, feature completeness, and consistency between the ML dataset and DL metadata.

The validation also confirms that the 09 previously identified boundary windows remain consistently excluded from both modelling pathways. No train/test split is performed at this stage; participant-level separation will be handled during model development to prevent data leakage.

In [ ]:
# Cell 67 — Final ML/DL dataset consistency validation

print("Final ML/DL dataset consistency validation")
print("=" * 60)



ml_window_ids = set(
    ml_dataset_df["WindowID"]
)

dl_window_ids = set(
    dl_metadata_df["WindowID"]
)

common_window_ids = (
    ml_window_ids
    &
    dl_window_ids
)

ml_only_ids = (
    ml_window_ids
    -
    dl_window_ids
)

dl_only_ids = (
    dl_window_ids
    -
    ml_window_ids
)

print("Window consistency:")
print(
    f"ML windows:       {len(ml_window_ids):,}"
)

print(
    f"DL windows:       {len(dl_window_ids):,}"
)

print(
    f"Common windows:   {len(common_window_ids):,}"
)

print(
    f"ML-only windows:  {len(ml_only_ids):,}"
)

print(
    f"DL-only windows:  {len(dl_only_ids):,}"
)



ml_participants = set(
    ml_dataset_df["ParticipantID"]
)

dl_participants = set(
    dl_metadata_df["ParticipantID"]
)

print("\nParticipant consistency:")

print(
    f"ML participants:  {len(ml_participants):,}"
)

print(
    f"DL participants:  {len(dl_participants):,}"
)

print(
    f"Participant sets identical: "
    f"{ml_participants == dl_participants}"
)



ml_labels = (
    ml_dataset_df[
        ["WindowID", "WakeSleepLabel"]
    ]
)

dl_labels = (
    dl_metadata_df[
        ["WindowID", "WakeSleepLabel"]
    ]
)

label_comparison = ml_labels.merge(
    dl_labels,
    on="WindowID",
    suffixes=("_ML", "_DL")
)

label_mismatches = (
    label_comparison["WakeSleepLabel_ML"]
    !=
    label_comparison["WakeSleepLabel_DL"]
).sum()

print("\nLabel consistency:")

print(
    f"Matched labelled windows: "
    f"{len(label_comparison):,}"
)

print(
    f"Label mismatches: "
    f"{label_mismatches:,}"
)


# Final consistency result


consistency_passed = (
    ml_window_ids == dl_window_ids
    and ml_participants == dl_participants
    and label_mismatches == 0
)

print("\nConsistency validation result:")
print(
    "PASSED"
    if consistency_passed
    else
    "REVIEW REQUIRED"
)

Final ML/DL dataset consistency validation
Window consistency:
ML windows:       421,166
DL windows:       421,166
Common windows:   421,166
ML-only windows:  0
DL-only windows:  0

Participant consistency:
ML participants:  30
DL participants:  30
Participant sets identical: True

Label consistency:
Matched labelled windows: 421,166
Label mismatches: 0

Consistency validation result:
PASSED


In [ ]:
# Cell 68 — Final dataset integrity validation

print("Final dataset integrity validation")
print("=" * 60)


# ML dataset integrity


ml_features = ml_dataset_df[
    ml_feature_columns
].to_numpy(dtype=np.float64)

ml_integrity = {
    "Rows": len(ml_dataset_df),
    "Missing feature values": int(
        np.isnan(ml_features).sum()
    ),
    "Infinite feature values": int(
        np.isinf(ml_features).sum()
    ),
    "Missing labels": int(
        ml_dataset_df["WakeSleepLabel"].isna().sum()
    ),
    "Duplicate WindowIDs": int(
        ml_dataset_df["WindowID"].duplicated().sum()
    ),
    "Participants": int(
        ml_dataset_df["ParticipantID"].nunique()
    )
}

print("ML dataset:")
for key, value in ml_integrity.items():
    print(f"{key}: {value:,}")


# DL metadata integrity


dl_integrity = {
    "Rows": len(dl_metadata_df),
    "Missing source paths": int(
        dl_metadata_df["Path"].isna().sum()
    ),
    "Missing timestamps": int(
        dl_metadata_df[
            "ActualFirstTimestamp"
        ].isna().sum()
    ),
    "Missing labels": int(
        dl_metadata_df["WakeSleepLabel"].isna().sum()
    ),
    "Duplicate WindowIDs": int(
        dl_metadata_df["WindowID"].duplicated().sum()
    ),
    "Participants": int(
        dl_metadata_df["ParticipantID"].nunique()
    ),
    "Invalid sample indices": int(
        (dl_metadata_df["StartSample"] < 0).sum()
    )
}

print("\nDL dataset metadata:")
for key, value in dl_integrity.items():
    print(f"{key}: {value:,}")


# Final class distribution


print("\nFinal class distribution:")

final_class_distribution = (
    ml_dataset_df["WakeSleepLabel"]
    .value_counts()
    .sort_index()
)

display(
    final_class_distribution
    .rename_axis("WakeSleepLabel")
    .reset_index(name="Count")
)


# Overall validation


final_validation_passed = (
    ml_integrity["Missing feature values"] == 0
    and ml_integrity["Infinite feature values"] == 0
    and ml_integrity["Missing labels"] == 0
    and ml_integrity["Duplicate WindowIDs"] == 0
    and ml_integrity["Participants"] == 30
    and dl_integrity["Missing source paths"] == 0
    and dl_integrity["Missing timestamps"] == 0
    and dl_integrity["Missing labels"] == 0
    and dl_integrity["Duplicate WindowIDs"] == 0
    and dl_integrity["Participants"] == 30
    and dl_integrity["Invalid sample indices"] == 0
    and len(ml_dataset_df) == len(dl_metadata_df)
)

print("\nFinal validation result:")
print(
    "PASSED"
    if final_validation_passed
    else
    "REVIEW REQUIRED"
)

Final dataset integrity validation
ML dataset:
Rows: 421,166
Missing feature values: 0
Infinite feature values: 0
Missing labels: 0
Duplicate WindowIDs: 0
Participants: 30

DL dataset metadata:
Rows: 421,166
Missing source paths: 0
Missing timestamps: 0
Missing labels: 0
Duplicate WindowIDs: 0
Participants: 30
Invalid sample indices: 0

Final class distribution:


,WakeSleepLabel,Count
0,0,87407
1,1,333759



Final validation result:
PASSED


## 13. Export for Modelling

The validated ML feature dataset and deep-learning window metadata are now prepared for export.

The ML dataset contains the engineered ECG features and corresponding Wake/Sleep labels required for traditional machine-learning models. The DL metadata contains the validated ECG window locations, labels, participant identifiers, and sample indices required to retrieve the corresponding 2,500-sample ECG waveforms.

Only validated windows are exported. The 09 boundary windows identified during timestamp validation remain excluded from the modelling datasets.

These exported datasets provide the final interface between Notebook 3 and the subsequent modelling notebooks.

In [ ]:
# Cell 70 — Export validated datasets for modelling

import os

print("Exporting validated datasets for modelling")
print("=" * 60)


# Create export directory


export_dir = (
    "/content/drive/MyDrive/"
    "QMUL_MSc_Dissertation/"
    "Notebook3_Exports"
)

os.makedirs(
    export_dir,
    exist_ok=True
)



ml_export_path = os.path.join(
    export_dir,
    "ML_feature_dataset.csv"
)

ml_dataset_df.to_csv(
    ml_export_path,
    index=False
)



dl_export_path = os.path.join(
    export_dir,
    "DL_window_metadata.csv"
)

dl_metadata_df.to_csv(
    dl_export_path,
    index=False
)



feature_list_path = os.path.join(
    export_dir,
    "feature_columns.txt"
)

with open(
    feature_list_path,
    "w"
) as f:

    for feature in ml_feature_columns:
        f.write(feature + "\n")


# Confirm files exist

print("Exported files:")
print("-" * 60)

for path in [
    ml_export_path,
    dl_export_path,
    feature_list_path
]:

    if os.path.exists(path):

        size_mb = (
            os.path.getsize(path)
            / (1024 ** 2)
        )

        print(
            f"{os.path.basename(path)}"
            f"  |  {size_mb:.2f} MB"
        )

    else:
        print(
            f"{os.path.basename(path)}"
            f"  |  NOT FOUND"
        )

# Final export validation


ml_export_exists = os.path.exists(
    ml_export_path
)

dl_export_exists = os.path.exists(
    dl_export_path
)

feature_list_exists = os.path.exists(
    feature_list_path
)

export_passed = (
    ml_export_exists
    and dl_export_exists
    and feature_list_exists
)

print("\nExport validation:")
print(
    "PASSED"
    if export_passed
    else
    "FAILED"
)

if export_passed:
    print(
        "\nNotebook 3 modelling datasets "
        "exported successfully."
    )

Exporting validated datasets for modelling
Exported files:
------------------------------------------------------------
ML_feature_dataset.csv  |  76.16 MB
DL_window_metadata.csv  |  81.37 MB
feature_columns.txt  |  0.00 MB

Export validation:
PASSED

Notebook 3 modelling datasets exported successfully.


## 14. Final Dataset Summary

The completed Notebook 3 pipeline produces the final modelling datasets after Wake/Sleep label assignment, quality-aware ECG window selection, feature extraction, and validation.

### Final Dataset Statistics

- **Quality-aware candidate ECG windows:** 530,921
- **Final Wake/Sleep labelled ECG windows:** 421,166
- **Participants represented:** 30
- **ECG recordings represented:** 34
- **Window duration:** 10 seconds
- **Window overlap:** 50%
- **Sampling frequency:** 250 Hz
- **Samples per window:** 2,500
- **Wake/Sleep matching tolerance:** ±15 minutes

### Dataset Consistency

The final ML and DL datasets contain the same set of validated ECG windows.

- **ML dataset windows:** 421,166
- **DL dataset windows:** 421,166
- **Common WindowIDs:** 421,166
- **Label mismatches between ML and DL datasets:** 0
- **Missing or invalid feature values:** 0

The final dataset therefore provides a consistent basis for subsequent model development.

> **Critical requirement for Notebook 4:** Because ECG windows overlap by 50%, adjacent windows from the same participant are highly correlated, and participant-specific ECG characteristics exist, Notebook 4 **must not** randomly split individual ECG windows across train/validation/test sets. Every participant must belong entirely to one split or fold (e.g. via `GroupKFold` on `ParticipantID`) to prevent data leakage.

### Final Methodological Status

The final ECG windows were generated only within continuous quality-valid signal segments. Wake/Sleep labels were assigned using the nearest available reference observation within the scientifically justified ±15-minute tolerance. The resulting labelled dataset contains **421,166 ECG windows** and is exported for subsequent traditional machine-learning and deep-learning experiments.

### Final Methodological Decisions

- ECG windows were restricted to continuous quality-valid signal segments.
- A 10-second window with 50% overlap was retained.
- Wake/Sleep labels were assigned using the nearest reference observation within a ±15-minute tolerance.
- The ±15-minute tolerance was retained based on the observed temporal resolution of the Wake/Sleep reference data and the transition-sensitivity analysis.
- The final quality-aware labelled dataset contains 421,166 ECG windows.
- ML and DL datasets were validated against the same WindowID set with no label mismatches.
- Participant-level separation will be applied during subsequent model development to reduce the risk of information leakage between training, validation, and test sets.